# Mapping the Milky Way in Six Dimensions: A contiguous, homogenised, phase-space catalogue of $\textit{Gaia}$ DR3 tracers up to 250 kpc

Paper: https://arxiv.org/abs/2609.28652

# Part 1

In [ ]:
# ============================================================
# MAPPING THE MILKY WAY IN 6D
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–3 s      Solar neighbourhood
#   3–7 s      Galactic disk appears
#   7–10.5 s   Stellar halo extends outward
#   10.5–13.2  Velocity vectors reveal the extra 3 dimensions
#   13.2–14 s  Transition
#   14–20 s    Final catalogue summary
#
# IMPORTANT
# The stellar distribution and velocity vectors are SCHEMATIC.
# They are NOT individual catalogue objects.
#
# Numerical catalogue statistics on the final card
# are taken from the paper.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "mapping_milky_way_6d.webm"
else:
    OUTFILE = "mapping_milky_way_6d.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"

rng = np.random.default_rng(42)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "MAPPING THE MILKY WAY IN 6D",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Position + motion across the Galactic disk and halo",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# MAIN GALAXY PANEL
# ============================================================

ax = fig.add_axes(
    [0.055, 0.13, 0.70, 0.72],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)

ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# GALACTIC BACKGROUND GLOWS
# ============================================================

halo_glow = Circle(
    (0, 0),
    0.92,
    facecolor="#3A6480",
    edgecolor="none",
    alpha=0,
    zorder=1
)

disk_glow_outer = Ellipse(
    (0, 0),
    1.35,
    0.43,
    angle=-8,
    facecolor="#79B8D1",
    edgecolor="none",
    alpha=0,
    zorder=2
)

disk_glow_inner = Ellipse(
    (0, 0),
    0.75,
    0.25,
    angle=-8,
    facecolor="#D4EAF2",
    edgecolor="none",
    alpha=0,
    zorder=3
)

bulge = Ellipse(
    (0, 0),
    0.20,
    0.14,
    angle=-8,
    facecolor="#E8D6A2",
    edgecolor="none",
    alpha=0,
    zorder=4
)

ax.add_patch(halo_glow)
ax.add_patch(disk_glow_outer)
ax.add_patch(disk_glow_inner)
ax.add_patch(bulge)


# ============================================================
# SCHEMATIC STELLAR POPULATIONS
# ============================================================

# ------------------------------------------------------------
# LOCAL NEIGHBOURHOOD
# ------------------------------------------------------------

N_LOCAL = 115

local_r = (
    0.11 *
    np.sqrt(
        rng.random(N_LOCAL)
    )
)

local_phi = (
    rng.uniform(
        0,
        2*np.pi,
        N_LOCAL
    )
)

local_x = (
    0.27 +
    local_r *
    np.cos(local_phi)
)

local_y = (
    -0.03 +
    local_r *
    np.sin(local_phi)
)

local_sizes = rng.uniform(
    7,
    18,
    N_LOCAL
)

local_scatter = ax.scatter(
    local_x,
    local_y,
    s=local_sizes,
    c="#DDF7FF",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ------------------------------------------------------------
# DISK STARS
# ------------------------------------------------------------

N_DISK = 1250

disk_r = (
    rng.gamma(
        shape=2.0,
        scale=0.19,
        size=N_DISK
    )
)

disk_r = np.clip(
    disk_r,
    0.03,
    0.69
)

disk_phi = rng.uniform(
    0,
    2*np.pi,
    N_DISK
)

disk_x0 = (
    disk_r *
    np.cos(disk_phi)
)

disk_y0 = (
    0.26 *
    disk_r *
    np.sin(disk_phi)
)

# rotate projected disk
angle = np.deg2rad(-8)

disk_x = (
    disk_x0 * np.cos(angle) -
    disk_y0 * np.sin(angle)
)

disk_y = (
    disk_x0 * np.sin(angle) +
    disk_y0 * np.cos(angle)
)

disk_sizes = rng.uniform(
    1.5,
    7.0,
    N_DISK
)

disk_scatter = ax.scatter(
    disk_x,
    disk_y,
    s=disk_sizes,
    c="#BDE9F7",
    alpha=0,
    linewidths=0,
    zorder=10
)


# ------------------------------------------------------------
# HALO TRACERS
# ------------------------------------------------------------

N_HALO = 850

# radially extended, centrally concentrated halo
u = rng.random(N_HALO)

halo_r = (
    0.18 +
    0.77 *
    u**0.55
)

halo_phi = rng.uniform(
    0,
    2*np.pi,
    N_HALO
)

halo_flattening = 0.78

halo_x = (
    halo_r *
    np.cos(halo_phi)
)

halo_y = (
    halo_flattening *
    halo_r *
    np.sin(halo_phi)
)

halo_sizes = rng.uniform(
    1.5,
    5.0,
    N_HALO
)

halo_scatter = ax.scatter(
    halo_x,
    halo_y,
    s=halo_sizes,
    c="#83BCD2",
    alpha=0,
    linewidths=0,
    zorder=6
)


# ------------------------------------------------------------
# SCHEMATIC STELLAR STREAM
# ------------------------------------------------------------

N_STREAM = 190

stream_theta = np.linspace(
    -1.15,
    1.45,
    N_STREAM
)

stream_r = (
    0.70 +
    0.055 *
    np.sin(
        2.4 *
        stream_theta
    )
)

stream_x = (
    stream_r *
    np.cos(stream_theta)
)

stream_y = (
    0.75 *
    stream_r *
    np.sin(stream_theta)
)

stream_x += rng.normal(
    0,
    0.012,
    N_STREAM
)

stream_y += rng.normal(
    0,
    0.012,
    N_STREAM
)

stream_scatter = ax.scatter(
    stream_x,
    stream_y,
    s=rng.uniform(
        2.5,
        6.0,
        N_STREAM
    ),
    c="#A9E6F6",
    alpha=0,
    linewidths=0,
    zorder=8
)


# ============================================================
# SUN
# ============================================================

sun_glow = Circle(
    (0.27, -0.03),
    0.037,
    facecolor="#FFE29A",
    edgecolor="none",
    alpha=0.15,
    zorder=21
)

sun = Circle(
    (0.27, -0.03),
    0.011,
    facecolor="#FFF2C0",
    edgecolor="#FFFFFF",
    linewidth=0.8,
    alpha=1,
    zorder=22
)

ax.add_patch(
    sun_glow
)

ax.add_patch(
    sun
)

sun_label = ax.text(
    0.27,
    -0.105,
    "SUN",
    ha="center",
    va="center",
    color="#FFF0BE",
    fontsize=9,
    fontweight="bold",
    zorder=25
)


# ============================================================
# SCALE RINGS
# ============================================================

scale_ring_1 = Circle(
    (0.27, -0.03),
    0.115,
    facecolor="none",
    edgecolor="#89DDF7",
    linewidth=1.0,
    linestyle="--",
    alpha=0,
    zorder=18
)

scale_ring_2 = Circle(
    (0, 0),
    0.42,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.9,
    linestyle="--",
    alpha=0,
    zorder=5
)

scale_ring_3 = Circle(
    (0, 0),
    0.90,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.9,
    linestyle="--",
    alpha=0,
    zorder=5
)

ax.add_patch(scale_ring_1)
ax.add_patch(scale_ring_2)
ax.add_patch(scale_ring_3)


scale_text_1 = ax.text(
    0.27,
    0.115,
    "LOCAL",
    ha="center",
    va="center",
    color="#8EDFF7",
    fontsize=8,
    fontweight="bold",
    alpha=0,
    zorder=25
)

scale_text_2 = ax.text(
    -0.43,
    0.04,
    "DISK",
    ha="center",
    va="center",
    color="#8EAAB8",
    fontsize=8,
    fontweight="bold",
    alpha=0,
    zorder=25
)

scale_text_3 = ax.text(
    -0.72,
    0.61,
    "HALO",
    ha="center",
    va="center",
    color="#8EAAB8",
    fontsize=8,
    fontweight="bold",
    alpha=0,
    zorder=25
)


# ============================================================
# VELOCITY VECTORS
# ============================================================

# Selected stars only, to keep the frame readable.
N_VEC_DISK = 48

vec_idx_disk = rng.choice(
    N_DISK,
    N_VEC_DISK,
    replace=False
)

vx_disk = (
    -disk_y[vec_idx_disk]
)

vy_disk = (
    disk_x[vec_idx_disk]
)

norm = np.sqrt(
    vx_disk**2 +
    vy_disk**2
) + 1e-9

vx_disk = (
    vx_disk /
    norm *
    0.055
)

vy_disk = (
    vy_disk /
    norm *
    0.055
)


disk_quiver = ax.quiver(
    disk_x[vec_idx_disk],
    disk_y[vec_idx_disk],
    vx_disk,
    vy_disk,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#B9F1FF",
    width=0.0032,
    headwidth=3.8,
    headlength=4.8,
    alpha=0,
    zorder=30
)


N_VEC_HALO = 32

vec_idx_halo = rng.choice(
    N_HALO,
    N_VEC_HALO,
    replace=False
)

# random schematic halo motions
vec_angle = rng.uniform(
    0,
    2*np.pi,
    N_VEC_HALO
)

vec_len = rng.uniform(
    0.035,
    0.070,
    N_VEC_HALO
)

vx_halo = (
    vec_len *
    np.cos(vec_angle)
)

vy_halo = (
    vec_len *
    np.sin(vec_angle)
)


halo_quiver = ax.quiver(
    halo_x[vec_idx_halo],
    halo_y[vec_idx_halo],
    vx_halo,
    vy_halo,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#75BCD7",
    width=0.0028,
    headwidth=3.6,
    headlength=4.6,
    alpha=0,
    zorder=29
)


# ============================================================
# RIGHT INFORMATION PANEL
# ============================================================

info_ax = fig.add_axes(
    [0.77, 0.20, 0.19, 0.55],
    zorder=15
)

info_ax.set_facecolor(
    "#050C13"
)

info_ax.set_xlim(
    0,
    1
)

info_ax.set_ylim(
    0,
    1
)

info_ax.set_xticks([])
info_ax.set_yticks([])

for spine in info_ax.spines.values():
    spine.set_color("#355466")


info_title = info_ax.text(
    0.5, 0.91,
    "GALACTIC MAP",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


# ------------------------------------------------------------
# 3D POSITION
# ------------------------------------------------------------

pos_title = info_ax.text(
    0.5, 0.74,
    "3D POSITION",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=12,
    fontweight="bold",
    alpha=0
)

pos_values = info_ax.text(
    0.5, 0.64,
    "X   Y   Z",
    ha="center",
    va="center",
    color="#D7E8EF",
    fontsize=12,
    alpha=0
)


# ------------------------------------------------------------
# PLUS
# ------------------------------------------------------------

plus = info_ax.text(
    0.5, 0.52,
    "+",
    ha="center",
    va="center",
    color="#78DFFF",
    fontsize=20,
    fontweight="bold",
    alpha=0
)


# ------------------------------------------------------------
# 3D VELOCITY
# ------------------------------------------------------------

vel_title = info_ax.text(
    0.5, 0.40,
    "3D VELOCITY",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=12,
    fontweight="bold",
    alpha=0
)

vel_values = info_ax.text(
    0.5, 0.30,
    "VX   VY   VZ",
    ha="center",
    va="center",
    color="#D7E8EF",
    fontsize=12,
    alpha=0
)


# ------------------------------------------------------------
# 6D
# ------------------------------------------------------------

sixd = info_ax.text(
    0.5, 0.13,
    "6D PHASE SPACE",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=13,
    fontweight="bold",
    alpha=0
)


# ============================================================
# PHASE LABEL
# ============================================================

phase_text = fig.text(
    0.39, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.39, 0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# DISTANCE / SCALE DISPLAY
# ============================================================

distance_box = fig.text(
    0.865, 0.805,
    "",
    ha="center",
    va="center",
    color="#DDF7FF",
    fontsize=12,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#07131E",
        edgecolor="#52788C",
        linewidth=1.0
    ),
    zorder=40
)

distance_patch = distance_box.get_bbox_patch()


# ============================================================
# FINAL DARK BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL GALAXY ICON
# ============================================================

final_galaxy_ax = fig.add_axes(
    [0.065, 0.275, 0.36, 0.47],
    zorder=80
)

final_galaxy_ax.set_facecolor(
    "#000207"
)

final_galaxy_ax.set_xlim(
    -1,
    1
)

final_galaxy_ax.set_ylim(
    -1,
    1
)

final_galaxy_ax.set_aspect("equal")
final_galaxy_ax.axis("off")
final_galaxy_ax.set_visible(False)


# final halo
fg_halo = Circle(
    (0, 0),
    0.90,
    facecolor="#31566F",
    edgecolor="#527D93",
    linewidth=0.8,
    alpha=0.10,
    zorder=1
)

fg_disk_outer = Ellipse(
    (0, 0),
    1.55,
    0.48,
    angle=-9,
    facecolor="#77BCD6",
    edgecolor="#BDEEFF",
    linewidth=1.0,
    alpha=0.22,
    zorder=3
)

fg_disk_inner = Ellipse(
    (0, 0),
    0.78,
    0.24,
    angle=-9,
    facecolor="#D3EAF2",
    edgecolor="none",
    alpha=0.25,
    zorder=4
)

fg_bulge = Ellipse(
    (0, 0),
    0.21,
    0.15,
    angle=-9,
    facecolor="#F0D89C",
    edgecolor="none",
    alpha=0.85,
    zorder=5
)

final_galaxy_ax.add_patch(
    fg_halo
)

final_galaxy_ax.add_patch(
    fg_disk_outer
)

final_galaxy_ax.add_patch(
    fg_disk_inner
)

final_galaxy_ax.add_patch(
    fg_bulge
)


# ------------------------------------------------------------
# Final schematic halo points
# ------------------------------------------------------------

fg_idx = rng.choice(
    N_HALO,
    280,
    replace=False
)

final_galaxy_ax.scatter(
    halo_x[fg_idx],
    halo_y[fg_idx],
    s=4,
    c="#8CC9DF",
    alpha=0.48,
    linewidths=0,
    zorder=2
)


fg_disk_idx = rng.choice(
    N_DISK,
    420,
    replace=False
)

final_galaxy_ax.scatter(
    disk_x[fg_disk_idx],
    disk_y[fg_disk_idx],
    s=3,
    c="#D0F0FA",
    alpha=0.58,
    linewidths=0,
    zorder=6
)


# ------------------------------------------------------------
# Final velocity arrows
# ------------------------------------------------------------

fg_vec_idx = rng.choice(
    N_HALO,
    22,
    replace=False
)

fg_ang = rng.uniform(
    0,
    2*np.pi,
    len(fg_vec_idx)
)

fg_len = rng.uniform(
    0.04,
    0.07,
    len(fg_vec_idx)
)

final_galaxy_ax.quiver(
    halo_x[fg_vec_idx],
    halo_y[fg_vec_idx],
    fg_len * np.cos(fg_ang),
    fg_len * np.sin(fg_ang),
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#9AE5FA",
    width=0.003,
    alpha=0.75,
    zorder=10
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.820,
    "A HOMOGENISED 6D MAP OF THE MILKY WAY",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_subheader = fig.text(
    0.5, 0.777,
    "Positions and velocities across the Galactic disk and halo",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_number = fig.text(
    0.68, 0.655,
    "32,552,876",
    ha="center",
    va="center",
    color="#C5F5FF",
    fontsize=27,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_number_label = fig.text(
    0.68, 0.602,
    "UNIQUE STARS",
    ha="center",
    va="center",
    color="#A3C7D7",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_6d = fig.text(
    0.68, 0.500,
    "FULL 6D PHASE SPACE",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=15,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_6d_sub = fig.text(
    0.68, 0.458,
    "X • Y • Z   +   VX • VY • VZ",
    ha="center",
    va="center",
    color="#91AEBB",
    fontsize=10,
    alpha=0,
    zorder=100
)


final_halo = fig.text(
    0.68, 0.355,
    "517,123 HALO TRACERS",
    ha="center",
    va="center",
    color="#B7EFFF",
    fontsize=14,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_distance = fig.text(
    0.68, 0.285,
    "OUT TO ~250 kpc",
    ha="center",
    va="center",
    color="#D8EDF5",
    fontsize=14,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_note = fig.text(
    0.68, 0.205,
    "Gaia DR3 + 14 major spectroscopic surveys",
    ha="center",
    va="center",
    color="#819FAC",
    fontsize=9,
    alpha=0,
    zorder=100
)


final_caveat = fig.text(
    0.5, 0.115,
    "SCHEMATIC VISUALISATION — POINTS SHOWN ARE NOT INDIVIDUAL CATALOGUE SOURCES",
    ha="center",
    va="center",
    color="#7795A4",
    fontsize=8.5,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only — NOT a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # PHASE 1 — LOCAL NEIGHBOURHOOD
    # ========================================================

    local_p = smoothstep(
        sec / 1.8
    )

    local_scatter.set_alpha(
        0.92 * local_p
    )

    scale_ring_1.set_alpha(
        0.65 * local_p
    )

    scale_text_1.set_alpha(
        local_p
    )


    # ========================================================
    # PHASE 2 — GALACTIC DISK
    # ========================================================

    disk_p = smoothstep(
        (sec - 2.6) / 2.8
    )

    disk_scatter.set_alpha(
        0.50 * disk_p
    )

    disk_glow_outer.set_alpha(
        0.065 * disk_p
    )

    disk_glow_inner.set_alpha(
        0.11 * disk_p
    )

    bulge.set_alpha(
        0.78 * disk_p
    )

    scale_ring_2.set_alpha(
        0.32 * disk_p
    )

    scale_text_2.set_alpha(
        0.75 * disk_p
    )


    # local region becomes less visually dominant
    local_fade = (
        1.0 -
        0.55 *
        smoothstep(
            (sec - 4.5) / 2.0
        )
    )

    local_scatter.set_alpha(
        0.92 *
        local_p *
        local_fade
    )


    # ========================================================
    # PHASE 3 — HALO
    # ========================================================

    halo_p = smoothstep(
        (sec - 6.5) / 2.5
    )

    halo_scatter.set_alpha(
        0.42 * halo_p
    )

    stream_scatter.set_alpha(
        0.60 *
        smoothstep(
            (sec - 7.7) / 1.6
        )
    )

    halo_glow.set_alpha(
        0.045 * halo_p
    )

    scale_ring_3.set_alpha(
        0.30 * halo_p
    )

    scale_text_3.set_alpha(
        0.75 * halo_p
    )


    # ========================================================
    # SUN GLOW
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 2.8
        )
    )

    sun_glow.set_alpha(
        0.11 +
        0.08 * pulse
    )


    # ========================================================
    # INFORMATION PANEL — POSITION
    # ========================================================

    position_p = smoothstep(
        (sec - 5.2) / 1.0
    )

    pos_title.set_alpha(
        position_p
    )

    pos_values.set_alpha(
        position_p
    )


    # ========================================================
    # VELOCITIES / 6D
    # ========================================================

    velocity_p = smoothstep(
        (sec - 10.0) / 1.8
    )

    plus.set_alpha(
        velocity_p
    )

    vel_title.set_alpha(
        velocity_p
    )

    vel_values.set_alpha(
        velocity_p
    )

    sixd.set_alpha(
        smoothstep(
            (sec - 11.2) / 1.0
        )
    )

    disk_quiver.set_alpha(
        0.82 * velocity_p
    )

    halo_quiver.set_alpha(
        0.68 * velocity_p
    )


    # ========================================================
    # PHASE TEXT + SCALE
    # ========================================================

    if sec < 3.0:

        phase_text.set_text(
            "THE SOLAR NEIGHBOURHOOD"
        )

        phase_sub.set_text(
            "Start with stars around the Sun"
        )

        scale_value = (
            0.1 +
            0.9 *
            smoothstep(
                sec / 3.0
            )
        )

        distance_box.set_text(
            f"SCALE  ~{scale_value:.1f} kpc"
        )

    elif sec < 7.0:

        phase_text.set_text(
            "ZOOM OUT TO THE GALACTIC DISK"
        )

        phase_sub.set_text(
            "Millions of stellar tracers reveal the structure of the Milky Way"
        )

        p = smoothstep(
            (sec - 3.0) / 4.0
        )

        scale_value = (
            1 +
            24 * p
        )

        distance_box.set_text(
            f"SCALE  ~{scale_value:.0f} kpc"
        )

    elif sec < 10.5:

        phase_text.set_text(
            "THE MAP EXTENDS INTO THE HALO"
        )

        phase_sub.set_text(
            "Distant tracers reach far beyond the stellar disk"
        )

        p = smoothstep(
            (sec - 7.0) / 3.5
        )

        scale_value = (
            25 +
            225 * p
        )

        distance_box.set_text(
            f"SCALE  ~{scale_value:.0f} kpc"
        )

    else:

        phase_text.set_text(
            "NOW ADD MOTION"
        )

        phase_sub.set_text(
            "Three coordinates of position + three coordinates of velocity"
        )

        distance_box.set_text(
            "FULL 6D MAP"
        )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )

    distance_box.set_alpha(
        scene1_alpha
    )

    distance_patch.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.98 * scene2
        )

        final_galaxy_ax.set_visible(
            True
        )

        final_galaxy_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_galaxy_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_subheader.set_alpha(
        scene2
    )

    final_number.set_alpha(
        scene2
    )

    final_number_label.set_alpha(
        scene2
    )

    final_6d.set_alpha(
        scene2
    )

    final_6d_sub.set_alpha(
        scene2
    )

    final_halo.set_alpha(
        scene2
    )

    final_distance.set_alpha(
        scene2
    )

    final_note.set_alpha(
        scene2
    )

    final_caveat.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Карта Млечного Пути в шести измерениях

Чтобы описать положение звезды в Галактике, достаточно трёх пространственных координат. Но чтобы понять её движение и восстановить динамическую историю Млечного Пути, нужны ещё три компоненты скорости. Вместе они образуют шестимерное фазовое пространство — 6D.

Авторы новой работы объединили данные Gaia DR3 и 14 крупных спектроскопических обзоров, сведя более 52 миллионов исходных записей в однородный каталог из 32 552 876 уникальных звёзд с полными шестимерными координатами. 2609.28652v1.pdf

Карта охватывает не только окрестности Солнца и диск Млечного Пути. Она простирается примерно до 250 килопарсек и содержит 517 123 звёздных трассера гало, позволяя исследовать области далеко за пределами основной видимой части Галактики. 2609.28652v1.pdf

Такая карта показывает уже не просто то, где находятся звёзды, но и куда они движутся. Это позволяет искать кинематические структуры, звёздные потоки и остатки древних событий слияния, исследовать галактическое гало и восстанавливать динамическую историю Млечного Пути. Авторы проверяли полученный каталог в том числе на звёздных скоплениях, карликовых галактиках и потоке Стрельца.

Распределение точек и векторы движения в анимации схематические и не представляют отдельные объекты нового каталога. Численность каталога, количество трассеров гало и его пространственный масштаб соответствуют данным статьи.

# Mapping the Milky Way in Six Dimensions

Three spatial coordinates are sufficient to describe where a star is located in the Galaxy. But understanding its motion — and reconstructing the dynamical history of the Milky Way — requires another three velocity components. Together they form six-dimensional phase space, or 6D.

The authors of the new study combined Gaia DR3 with 14 major spectroscopic surveys, transforming more than 52 million input records into a homogenised catalogue containing 32,552,876 unique stars with complete six-dimensional phase-space information. 2609.28652v1.pdf

The resulting map extends far beyond the Solar neighbourhood and the Galactic disk. It reaches distances of approximately 250 kiloparsecs and contains 517,123 stellar halo tracers, probing regions far outside the main visible body of the Milky Way. 2609.28652v1.pdf

Such a catalogue tells us not only where stars are, but also where they are going. This makes it possible to identify kinematic structures and stellar streams, investigate the Galactic halo and remnants of past accretion events, and reconstruct the dynamical history of the Milky Way. The authors validate the catalogue using stellar clusters, dwarf galaxies, and the Sagittarius stream, among other tests.

The stellar distribution and velocity vectors shown in the animation are schematic and do not represent individual sources from the new catalogue. The catalogue size, number of halo tracers, and spatial extent correspond to values reported in the study.

---

# Part 2

In [ ]:
# ============================================================
# THE GALAXY IS NOT STATIC — 6D PHASE SPACE
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4 s      Static 3D map: where are the stars?
#   4–8 s      Add stellar velocities
#   8–13.2 s   Disk / halo / stellar stream separate kinematically
#   13.2–14 s  Transition
#   14–20 s    Final 3D vs 6D comparison
#
# IMPORTANT
# All stars, velocities and structures shown here are SCHEMATIC.
# This is a conceptual illustration of phase-space information,
# not an animation of individual catalogue sources or measured orbits.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "milky_way_6d_phase_space.webm"
else:
    OUTFILE = "milky_way_6d_phase_space.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"

rng = np.random.default_rng(73)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "THE GALAXY IS NOT STATIC",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "6D phase space adds stellar motion to the Galactic map",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# GALAXY PANEL
# ============================================================

ax = fig.add_axes(
    [0.045, 0.16, 0.67, 0.69],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -1,
    1
)

ax.set_ylim(
    -1,
    1
)

ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# BACKGROUND GALAXY
# ============================================================

halo_glow = Circle(
    (0, 0),
    0.91,
    facecolor="#355E78",
    edgecolor="none",
    alpha=0.045,
    zorder=1
)

disk_glow = Ellipse(
    (0, 0),
    1.45,
    0.47,
    angle=-8,
    facecolor="#75BBD5",
    edgecolor="none",
    alpha=0.065,
    zorder=2
)

disk_inner = Ellipse(
    (0, 0),
    0.73,
    0.24,
    angle=-8,
    facecolor="#D5EAF1",
    edgecolor="none",
    alpha=0.09,
    zorder=3
)

bulge = Ellipse(
    (0, 0),
    0.20,
    0.14,
    angle=-8,
    facecolor="#EAD49B",
    edgecolor="none",
    alpha=0.78,
    zorder=4
)

ax.add_patch(halo_glow)
ax.add_patch(disk_glow)
ax.add_patch(disk_inner)
ax.add_patch(bulge)


# ============================================================
# DISK POPULATION
# ============================================================

N_DISK = 950

disk_r = rng.gamma(
    2.0,
    0.19,
    N_DISK
)

disk_r = np.clip(
    disk_r,
    0.04,
    0.70
)

disk_phi = rng.uniform(
    0,
    2*np.pi,
    N_DISK
)

x0 = (
    disk_r *
    np.cos(disk_phi)
)

y0 = (
    0.27 *
    disk_r *
    np.sin(disk_phi)
)

rot = np.deg2rad(-8)

disk_x = (
    x0 * np.cos(rot) -
    y0 * np.sin(rot)
)

disk_y = (
    x0 * np.sin(rot) +
    y0 * np.cos(rot)
)

disk_scatter = ax.scatter(
    disk_x,
    disk_y,
    s=rng.uniform(
        1.8,
        6.5,
        N_DISK
    ),
    c="#C4EAF5",
    alpha=0.48,
    linewidths=0,
    zorder=10
)


# ============================================================
# HALO POPULATION
# ============================================================

N_HALO = 520

halo_r = (
    0.20 +
    0.72 *
    rng.random(N_HALO)**0.58
)

halo_phi = rng.uniform(
    0,
    2*np.pi,
    N_HALO
)

halo_x = (
    halo_r *
    np.cos(halo_phi)
)

halo_y = (
    0.79 *
    halo_r *
    np.sin(halo_phi)
)

halo_scatter = ax.scatter(
    halo_x,
    halo_y,
    s=rng.uniform(
        1.8,
        5.0,
        N_HALO
    ),
    c="#729EB3",
    alpha=0.34,
    linewidths=0,
    zorder=6
)


# ============================================================
# SCHEMATIC STELLAR STREAM
# ============================================================

N_STREAM = 150

stream_t = np.linspace(
    -1.18,
    1.38,
    N_STREAM
)

stream_r = (
    0.72 +
    0.045 *
    np.sin(
        2.5 *
        stream_t
    )
)

stream_x = (
    stream_r *
    np.cos(stream_t)
)

stream_y = (
    0.76 *
    stream_r *
    np.sin(stream_t)
)

stream_x += rng.normal(
    0,
    0.009,
    N_STREAM
)

stream_y += rng.normal(
    0,
    0.009,
    N_STREAM
)

stream_scatter = ax.scatter(
    stream_x,
    stream_y,
    s=rng.uniform(
        3,
        6,
        N_STREAM
    ),
    c="#A6E8FA",
    alpha=0.53,
    linewidths=0,
    zorder=12
)


# ============================================================
# SUN
# ============================================================

sun_glow = Circle(
    (0.27, -0.03),
    0.037,
    facecolor="#FFE39B",
    edgecolor="none",
    alpha=0.15,
    zorder=20
)

sun = Circle(
    (0.27, -0.03),
    0.011,
    facecolor="#FFF2BF",
    edgecolor="white",
    linewidth=0.8,
    zorder=21
)

ax.add_patch(sun_glow)
ax.add_patch(sun)

sun_text = ax.text(
    0.27,
    -0.095,
    "SUN",
    ha="center",
    va="center",
    color="#FFEFC0",
    fontsize=8,
    fontweight="bold",
    zorder=22
)


# ============================================================
# VELOCITY VECTORS — DISK
# ============================================================

N_DV = 62

disk_idx = rng.choice(
    N_DISK,
    N_DV,
    replace=False
)

# Tangential schematic velocity field
dvx = (
    -disk_y[disk_idx]
)

dvy = (
    disk_x[disk_idx]
)

dvnorm = (
    np.sqrt(
        dvx**2 +
        dvy**2
    ) +
    1e-8
)

dvx = (
    dvx /
    dvnorm *
    0.060
)

dvy = (
    dvy /
    dvnorm *
    0.060
)

disk_quiver = ax.quiver(
    disk_x[disk_idx],
    disk_y[disk_idx],
    dvx,
    dvy,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#B9F3FF",
    width=0.0030,
    headwidth=3.7,
    headlength=4.7,
    alpha=0,
    zorder=30
)


# ============================================================
# VELOCITY VECTORS — HALO
# ============================================================

N_HV = 38

halo_idx = rng.choice(
    N_HALO,
    N_HV,
    replace=False
)

hv_angle = rng.uniform(
    0,
    2*np.pi,
    N_HV
)

hv_length = rng.uniform(
    0.04,
    0.075,
    N_HV
)

hvx = (
    hv_length *
    np.cos(hv_angle)
)

hvy = (
    hv_length *
    np.sin(hv_angle)
)

halo_quiver = ax.quiver(
    halo_x[halo_idx],
    halo_y[halo_idx],
    hvx,
    hvy,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#709EB3",
    width=0.0028,
    headwidth=3.6,
    headlength=4.5,
    alpha=0,
    zorder=28
)


# ============================================================
# VELOCITY VECTORS — STREAM
# ============================================================

N_SV = 24

stream_idx = np.linspace(
    5,
    N_STREAM - 6,
    N_SV
).astype(int)

# tangent to stream
dx = np.gradient(stream_x)
dy = np.gradient(stream_y)

svx = dx[stream_idx]
svy = dy[stream_idx]

svnorm = (
    np.sqrt(
        svx**2 +
        svy**2
    ) +
    1e-8
)

svx = (
    svx /
    svnorm *
    0.065
)

svy = (
    svy /
    svnorm *
    0.065
)

stream_quiver = ax.quiver(
    stream_x[stream_idx],
    stream_y[stream_idx],
    svx,
    svy,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#D1F8FF",
    width=0.0033,
    headwidth=3.8,
    headlength=4.8,
    alpha=0,
    zorder=35
)


# ============================================================
# POPULATION LABELS
# ============================================================

disk_label = ax.text(
    -0.48,
    -0.29,
    "GALACTIC DISK",
    ha="center",
    va="center",
    color="#BDECF8",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=40
)

disk_label_sub = ax.text(
    -0.48,
    -0.35,
    "coherent rotation",
    ha="center",
    va="center",
    color="#789AAA",
    fontsize=8,
    alpha=0,
    zorder=40
)


halo_label = ax.text(
    -0.67,
    0.59,
    "STELLAR HALO",
    ha="center",
    va="center",
    color="#8FB5C6",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=40
)

halo_label_sub = ax.text(
    -0.67,
    0.53,
    "complex motions",
    ha="center",
    va="center",
    color="#6F8995",
    fontsize=8,
    alpha=0,
    zorder=40
)


stream_label = ax.text(
    0.56,
    0.60,
    "STELLAR STREAM",
    ha="center",
    va="center",
    color="#C7F5FF",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=40
)

stream_label_sub = ax.text(
    0.56,
    0.54,
    "coherent phase-space structure",
    ha="center",
    va="center",
    color="#87AEBE",
    fontsize=8,
    alpha=0,
    zorder=40
)


# ============================================================
# RIGHT INFORMATION PANEL
# ============================================================

info_ax = fig.add_axes(
    [0.75, 0.20, 0.21, 0.57],
    zorder=15
)

info_ax.set_facecolor(
    "#050C13"
)

info_ax.set_xlim(
    0,
    1
)

info_ax.set_ylim(
    0,
    1
)

info_ax.set_xticks([])
info_ax.set_yticks([])

for spine in info_ax.spines.values():
    spine.set_color("#355466")


info_title = info_ax.text(
    0.5, 0.91,
    "PHASE SPACE",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


# ============================================================
# 3D
# ============================================================

three_d = info_ax.text(
    0.5, 0.76,
    "3D",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=19,
    fontweight="bold"
)

three_d_sub = info_ax.text(
    0.5, 0.68,
    "WHERE IS IT?",
    ha="center",
    va="center",
    color="#9DB7C4",
    fontsize=9,
    fontweight="bold"
)

three_d_coords = info_ax.text(
    0.5, 0.58,
    "X   Y   Z",
    ha="center",
    va="center",
    color="#D8E8EF",
    fontsize=13
)


# ============================================================
# ARROW
# ============================================================

info_arrow = info_ax.text(
    0.5, 0.46,
    "↓",
    ha="center",
    va="center",
    color="#73DFFF",
    fontsize=20,
    fontweight="bold",
    alpha=0
)


# ============================================================
# 6D
# ============================================================

six_d = info_ax.text(
    0.5, 0.34,
    "6D",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=21,
    fontweight="bold",
    alpha=0
)

six_d_sub = info_ax.text(
    0.5, 0.26,
    "WHERE IS IT GOING?",
    ha="center",
    va="center",
    color="#A5C6D5",
    fontsize=9,
    fontweight="bold",
    alpha=0
)

six_d_coords = info_ax.text(
    0.5, 0.15,
    "X Y Z  +  VX VY VZ",
    ha="center",
    va="center",
    color="#D8EEF5",
    fontsize=11,
    alpha=0
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.375, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.375, 0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL LEFT — STATIC MAP
# ============================================================

final_3d_ax = fig.add_axes(
    [0.07, 0.27, 0.36, 0.45],
    zorder=80
)

final_3d_ax.set_facecolor(
    "#030810"
)

final_3d_ax.set_xlim(
    -1,
    1
)

final_3d_ax.set_ylim(
    -1,
    1
)

final_3d_ax.set_aspect("equal")
final_3d_ax.axis("off")
final_3d_ax.set_visible(False)


f3_halo = Circle(
    (0, 0),
    0.86,
    facecolor="#31566F",
    edgecolor="none",
    alpha=0.07,
    zorder=1
)

f3_disk = Ellipse(
    (0, 0),
    1.55,
    0.48,
    angle=-8,
    facecolor="#74B9D2",
    edgecolor="#9CD7E9",
    linewidth=0.8,
    alpha=0.16,
    zorder=2
)

final_3d_ax.add_patch(f3_halo)
final_3d_ax.add_patch(f3_disk)


final_disk_idx = rng.choice(
    N_DISK,
    330,
    replace=False
)

final_halo_idx = rng.choice(
    N_HALO,
    150,
    replace=False
)

final_3d_ax.scatter(
    disk_x[final_disk_idx],
    disk_y[final_disk_idx],
    s=4,
    c="#C7EAF4",
    alpha=0.55,
    linewidths=0,
    zorder=5
)

final_3d_ax.scatter(
    halo_x[final_halo_idx],
    halo_y[final_halo_idx],
    s=4,
    c="#779FB1",
    alpha=0.40,
    linewidths=0,
    zorder=4
)


# ============================================================
# FINAL RIGHT — 6D MAP
# ============================================================

final_6d_ax = fig.add_axes(
    [0.57, 0.27, 0.36, 0.45],
    zorder=80
)

final_6d_ax.set_facecolor(
    "#030810"
)

final_6d_ax.set_xlim(
    -1,
    1
)

final_6d_ax.set_ylim(
    -1,
    1
)

final_6d_ax.set_aspect("equal")
final_6d_ax.axis("off")
final_6d_ax.set_visible(False)


f6_halo = Circle(
    (0, 0),
    0.86,
    facecolor="#31566F",
    edgecolor="none",
    alpha=0.07,
    zorder=1
)

f6_disk = Ellipse(
    (0, 0),
    1.55,
    0.48,
    angle=-8,
    facecolor="#74B9D2",
    edgecolor="#9CD7E9",
    linewidth=0.8,
    alpha=0.16,
    zorder=2
)

final_6d_ax.add_patch(f6_halo)
final_6d_ax.add_patch(f6_disk)


final_6d_ax.scatter(
    disk_x[final_disk_idx],
    disk_y[final_disk_idx],
    s=4,
    c="#C7EAF4",
    alpha=0.55,
    linewidths=0,
    zorder=5
)

final_6d_ax.scatter(
    halo_x[final_halo_idx],
    halo_y[final_halo_idx],
    s=4,
    c="#779FB1",
    alpha=0.40,
    linewidths=0,
    zorder=4
)


# velocity arrows in final 6D panel
final_vec_idx = rng.choice(
    N_DISK,
    38,
    replace=False
)

fvx = (
    -disk_y[final_vec_idx]
)

fvy = (
    disk_x[final_vec_idx]
)

fvn = (
    np.sqrt(
        fvx**2 +
        fvy**2
    ) +
    1e-8
)

fvx = (
    fvx /
    fvn *
    0.075
)

fvy = (
    fvy /
    fvn *
    0.075
)

final_6d_ax.quiver(
    disk_x[final_vec_idx],
    disk_y[final_vec_idx],
    fvx,
    fvy,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#C6F6FF",
    width=0.0032,
    headwidth=3.7,
    headlength=4.7,
    alpha=0.82,
    zorder=10
)


final_stream_idx = np.linspace(
    7,
    N_STREAM - 8,
    18
).astype(int)

fsx = dx[final_stream_idx]
fsy = dy[final_stream_idx]

fsn = (
    np.sqrt(
        fsx**2 +
        fsy**2
    ) +
    1e-8
)

fsx = (
    fsx /
    fsn *
    0.075
)

fsy = (
    fsy /
    fsn *
    0.075
)

final_6d_ax.scatter(
    stream_x,
    stream_y,
    s=4,
    c="#B5EFFF",
    alpha=0.65,
    linewidths=0,
    zorder=8
)

final_6d_ax.quiver(
    stream_x[final_stream_idx],
    stream_y[final_stream_idx],
    fsx,
    fsy,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="#E0FBFF",
    width=0.0034,
    headwidth=3.8,
    headlength=4.8,
    alpha=0.90,
    zorder=11
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.815,
    "POSITION BECOMES DYNAMICS",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_sub = fig.text(
    0.5, 0.770,
    "Six-dimensional phase space reveals structures hidden in a static map",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_3d_title = fig.text(
    0.25, 0.245,
    "3D POSITION",
    ha="center",
    va="center",
    color="#D9EDF5",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_3d_sub = fig.text(
    0.25, 0.205,
    "WHERE ARE THE STARS?",
    ha="center",
    va="center",
    color="#829DAA",
    fontsize=9,
    alpha=0,
    zorder=100
)


final_6d_title = fig.text(
    0.75, 0.245,
    "6D PHASE SPACE",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_6d_sub = fig.text(
    0.75, 0.205,
    "WHERE ARE THEY GOING?",
    ha="center",
    va="center",
    color="#91B4C3",
    fontsize=9,
    alpha=0,
    zorder=100
)


final_message = fig.text(
    0.5, 0.125,
    "DISK ROTATION • STELLAR STREAMS • HALO KINEMATICS • GALACTIC ARCHAEOLOGY",
    ha="center",
    va="center",
    color="#A8D9EA",
    fontsize=9.5,
    fontweight="bold",
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only — not a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(dial_outer)
clock_ax.add_patch(dial_inner)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(dial_pin)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 4.0:

        phase_text.set_text(
            "A STATIC MAP SHOWS POSITION"
        )

        phase_sub.set_text(
            "Three coordinates tell us where each star is"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "ADD THREE VELOCITY COMPONENTS"
        )

        phase_sub.set_text(
            "Every measured star becomes a point with motion"
        )

    else:

        phase_text.set_text(
            "KINEMATIC STRUCTURES EMERGE"
        )

        phase_sub.set_text(
            "Different Galactic populations occupy different regions of phase space"
        )


    # ========================================================
    # SUN PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 3.0
        )
    )

    sun_glow.set_alpha(
        0.11 +
        0.08 * pulse
    )


    # ========================================================
    # VELOCITIES APPEAR
    # ========================================================

    velocity_p = smoothstep(
        (sec - 4.0) / 2.3
    )

    disk_quiver.set_alpha(
        0.78 * velocity_p
    )

    halo_quiver.set_alpha(
        0.62 * velocity_p
    )

    stream_quiver.set_alpha(
        0.90 *
        smoothstep(
            (sec - 7.0) / 1.6
        )
    )


    # ========================================================
    # INFO PANEL
    # ========================================================

    info_arrow.set_alpha(
        velocity_p
    )

    six_d.set_alpha(
        velocity_p
    )

    six_d_sub.set_alpha(
        velocity_p
    )

    six_d_coords.set_alpha(
        velocity_p
    )


    # ========================================================
    # POPULATION LABELS
    # ========================================================

    populations_p = smoothstep(
        (sec - 8.0) / 1.5
    )

    disk_label.set_alpha(
        populations_p
    )

    disk_label_sub.set_alpha(
        populations_p
    )

    halo_label.set_alpha(
        populations_p
    )

    halo_label_sub.set_alpha(
        populations_p
    )

    stream_label.set_alpha(
        populations_p
    )

    stream_label_sub.set_alpha(
        populations_p
    )


    # ========================================================
    # SLIGHTLY EMPHASISE STREAM
    # ========================================================

    stream_emphasis = smoothstep(
        (sec - 9.0) / 1.5
    )

    stream_scatter.set_alpha(
        0.53 +
        0.25 * stream_emphasis
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(True)
        result_bg.set_alpha(
            0.98 * scene2
        )

        final_3d_ax.set_visible(True)
        final_3d_ax.set_alpha(
            scene2
        )

        final_6d_ax.set_visible(True)
        final_6d_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(False)
        final_3d_ax.set_visible(False)
        final_6d_ax.set_visible(False)


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_sub.set_alpha(
        scene2
    )

    final_3d_title.set_alpha(
        scene2
    )

    final_3d_sub.set_alpha(
        scene2
    )

    final_6d_title.set_alpha(
        scene2
    )

    final_6d_sub.set_alpha(
        scene2
    )

    final_message.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Млечный Путь не статичен: карта Галактики в 6D

Обычная трёхмерная карта отвечает на вопрос: где находится звезда? Три пространственные координаты определяют её положение в Галактике. Но сами по себе они почти ничего не говорят о том, откуда звезда пришла и куда движется.

Новый объединённый каталог добавляет к положению ещё три компоненты скорости. Получается полное шестимерное фазовое пространство: X, Y, Z + VX, VY, VZ. Каталог содержит более 32,5 миллиона уникальных звёзд с полной 6D-информацией, объединяя Gaia DR3 с измерениями 14 крупных спектроскопических обзоров. 2609.28652v1.pdf

Добавление скоростей превращает статическую карту в динамическую. Звёзды диска демонстрируют организованное галактическое вращение, тогда как население гало обладает значительно более сложной кинематикой. Группы звёзд, которые могут выглядеть разрозненными в обычном пространстве, способны образовывать связанные структуры в фазовом пространстве.

Особенно важны звёздные потоки — остатки разрушенных спутников и звёздных систем, растянутые гравитационным полем Млечного Пути. Совместный анализ положения и скорости позволяет выделять такие структуры и исследовать историю формирования Галактики. Авторы проверяли согласованность нового каталога на различных объектах, включая скопления, карликовые галактики и поток Стрельца.

Таким образом, 6D-карта показывает уже не просто распределение вещества в Млечном Пути. Она даёт возможность исследовать динамику Галактики, её звёздное гало, потоки и следы прошлых событий аккреции.

Распределение звёзд, поток и векторы скоростей в анимации схематические. Они демонстрируют принцип шестимерного фазового пространства и не являются визуализацией отдельных объектов нового каталога.

# The Milky Way Is Not Static: Mapping the Galaxy in 6D

A conventional three-dimensional map answers the question: where is a star? Three spatial coordinates determine its location in the Galaxy. But position alone tells us relatively little about where that star came from or where it is going.

The new homogenised catalogue adds three velocity components to the three spatial coordinates, producing complete six-dimensional phase space: X, Y, Z + VX, VY, VZ. It contains more than 32.5 million unique stars with full 6D information, combining Gaia DR3 with measurements from 14 major spectroscopic surveys. 2609.28652v1.pdf

Adding velocity transforms a static map into a dynamical one. Stars in the Galactic disk exhibit organised rotation, while halo populations possess much more complex kinematics. Groups of stars that may appear dispersed in ordinary space can remain identifiable as coherent structures in phase space.

This is particularly important for stellar streams — remnants of disrupted satellites and stellar systems stretched through the gravitational field of the Milky Way. Combining position and velocity makes such structures easier to identify and allows astronomers to investigate the assembly history of the Galaxy. The authors validate their catalogue using several classes of objects, including stellar clusters, dwarf galaxies, and the Sagittarius stream.

A 6D map therefore reveals more than the distribution of matter in the Milky Way. It provides a way to investigate Galactic dynamics, the stellar halo, stellar streams, and signatures of past accretion events.

The stellar distribution, stream, and velocity vectors shown in the animation are schematic. They illustrate the concept of six-dimensional phase space and do not represent individual objects from the new catalogue.

---

# Part 2

In [ ]:
# ============================================================
# FROM MANY SURVEYS TO ONE 6D GALAXY
#
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4.5 s    Gaia + spectroscopic surveys
#   4.5–8 s    Multiple measurements of the same stars
#   8–13.2 s   Cross-match → calibrate → deduplicate → combine
#   13.2–14 s  Transition
#   14–20 s    Final homogenised 6D catalogue
#
# IMPORTANT
# Data streams, individual stars and measurement offsets are
# SCHEMATIC. They illustrate the catalogue-building procedure.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, FancyBboxPatch
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "many_surveys_one_6d_catalogue.webm"
else:
    OUTFILE = "many_surveys_one_6d_catalogue.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"

rng = np.random.default_rng(112)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "FROM MANY SURVEYS TO ONE 6D GALAXY",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Cross-match • calibrate • deduplicate • combine",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# LEFT — SURVEYS
# ============================================================

survey_ax = fig.add_axes(
    [0.045, 0.20, 0.24, 0.62],
    zorder=10
)

survey_ax.set_facecolor(BG)
survey_ax.set_xlim(0, 1)
survey_ax.set_ylim(0, 1)
survey_ax.axis("off")


survey_header = survey_ax.text(
    0.5, 0.96,
    "INPUT SURVEYS",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


survey_names = [
    "GAIA DR3",
    "DESI",
    "SDSS",
    "LAMOST",
    "APOGEE",
    "+ 10 SURVEYS"
]

survey_y = np.array([
    0.82,
    0.69,
    0.56,
    0.43,
    0.30,
    0.17
])

survey_boxes = []
survey_texts = []


for i, (name, y) in enumerate(
    zip(
        survey_names,
        survey_y
    )
):

    box = FancyBboxPatch(
        (0.12, y - 0.043),
        0.76,
        0.086,
        boxstyle="round,pad=0.012,rounding_size=0.018",
        facecolor="#07131D",
        edgecolor="#50788C",
        linewidth=1.0,
        alpha=0,
        zorder=5
    )

    survey_ax.add_patch(
        box
    )

    txt = survey_ax.text(
        0.50,
        y,
        name,
        ha="center",
        va="center",
        color="#C5ECF8",
        fontsize=10,
        fontweight="bold",
        alpha=0,
        zorder=6
    )

    survey_boxes.append(
        box
    )

    survey_texts.append(
        txt
    )


# ============================================================
# DATA STREAMS
# ============================================================

stream_lines = []

for i, y in enumerate(survey_y):

    line, = survey_ax.plot(
        [0.88, 1.03],
        [y, 0.50],
        color="#79D8F2",
        lw=1.2,
        alpha=0,
        clip_on=False,
        zorder=4
    )

    stream_lines.append(
        line
    )


# ============================================================
# CENTER — SAME STAR, DIFFERENT MEASUREMENTS
# ============================================================

center_ax = fig.add_axes(
    [0.315, 0.20, 0.36, 0.62],
    zorder=12
)

center_ax.set_facecolor(
    "#040A11"
)

center_ax.set_xlim(
    -1,
    1
)

center_ax.set_ylim(
    -1,
    1
)

center_ax.set_aspect("equal")
center_ax.axis("off")


center_header = center_ax.text(
    0,
    0.91,
    "MULTIPLE MEASUREMENTS",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    alpha=0
)


# ------------------------------------------------------------
# True/reference schematic star
# ------------------------------------------------------------

reference_glow = Circle(
    (0, 0.36),
    0.12,
    facecolor="#FBE8A5",
    edgecolor="none",
    alpha=0,
    zorder=3
)

reference_star = Circle(
    (0, 0.36),
    0.037,
    facecolor="#FFF4C8",
    edgecolor="white",
    linewidth=1.0,
    alpha=0,
    zorder=5
)

center_ax.add_patch(
    reference_glow
)

center_ax.add_patch(
    reference_star
)


# ------------------------------------------------------------
# Schematic duplicate measurements around same star
# ------------------------------------------------------------

offsets = np.array([
    [-0.12,  0.04],
    [ 0.08,  0.07],
    [-0.06, -0.09],
    [ 0.13, -0.05],
    [ 0.02,  0.13],
    [-0.15, -0.03]
])

measurement_points = []

for i, (dx, dy) in enumerate(offsets):

    pt, = center_ax.plot(
        [dx],
        [0.36 + dy],
        marker="o",
        markersize=6,
        color="#83DCF4",
        alpha=0,
        zorder=8
    )

    measurement_points.append(
        pt
    )


# ------------------------------------------------------------
# Schematic radial-velocity measurements
# ------------------------------------------------------------

rv_y = np.array([
    0.05,
    -0.08,
    -0.21,
    -0.34,
    -0.47
])

rv_values = [
    "+126.4",
    "+124.9",
    "+127.1",
    "+125.6",
    "+126.0"
]

rv_labels = []

for y, value in zip(
    rv_y,
    rv_values
):

    txt = center_ax.text(
        0,
        y,
        f"RV   {value} km/s",
        ha="center",
        va="center",
        color="#91B8C8",
        fontsize=9,
        alpha=0,
        zorder=8
    )

    rv_labels.append(
        txt
    )


measurement_note = center_ax.text(
    0,
    -0.68,
    "SAME STAR • DIFFERENT SURVEYS",
    ha="center",
    va="center",
    color="#A8D9E9",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=8
)


# ============================================================
# RIGHT — PIPELINE
# ============================================================

pipe_ax = fig.add_axes(
    [0.70, 0.20, 0.26, 0.62],
    zorder=12
)

pipe_ax.set_facecolor(BG)
pipe_ax.set_xlim(0, 1)
pipe_ax.set_ylim(0, 1)
pipe_ax.axis("off")


pipe_header = pipe_ax.text(
    0.5, 0.96,
    "HOMOGENISATION",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    alpha=0
)


pipeline_names = [
    "CROSS-MATCH",
    "CALIBRATE",
    "DEDUPLICATE",
    "COMBINE"
]

pipeline_y = [
    0.78,
    0.59,
    0.40,
    0.21
]

pipeline_boxes = []
pipeline_texts = []
pipeline_arrows = []


for i, (name, y) in enumerate(
    zip(
        pipeline_names,
        pipeline_y
    )
):

    box = FancyBboxPatch(
        (0.14, y - 0.055),
        0.72,
        0.11,
        boxstyle="round,pad=0.015,rounding_size=0.022",
        facecolor="#07131D",
        edgecolor="#5B9BB5",
        linewidth=1.1,
        alpha=0,
        zorder=5
    )

    pipe_ax.add_patch(
        box
    )

    txt = pipe_ax.text(
        0.50,
        y,
        name,
        ha="center",
        va="center",
        color="#C8F2FF",
        fontsize=10.5,
        fontweight="bold",
        alpha=0,
        zorder=6
    )

    pipeline_boxes.append(
        box
    )

    pipeline_texts.append(
        txt
    )

    if i < len(pipeline_names) - 1:

        arrow = pipe_ax.text(
            0.50,
            y - 0.105,
            "↓",
            ha="center",
            va="center",
            color="#72DDF8",
            fontsize=17,
            fontweight="bold",
            alpha=0,
            zorder=7
        )

        pipeline_arrows.append(
            arrow
        )


# ============================================================
# INPUT / OUTPUT COUNTERS
# ============================================================

input_counter = fig.text(
    0.165,
    0.135,
    "52+ MILLION INPUT RECORDS",
    ha="center",
    va="center",
    color="#A4C7D6",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=50
)


output_counter = fig.text(
    0.83,
    0.135,
    "32,552,876 UNIQUE STARS",
    ha="center",
    va="center",
    color="#BDF3FF",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=50
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.5, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.5, 0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL GALAXY
# ============================================================

final_ax = fig.add_axes(
    [0.055, 0.25, 0.38, 0.49],
    zorder=80
)

final_ax.set_facecolor(
    "#000207"
)

final_ax.set_xlim(
    -1,
    1
)

final_ax.set_ylim(
    -1,
    1
)

final_ax.set_aspect("equal")
final_ax.axis("off")
final_ax.set_visible(False)


# ------------------------------------------------------------
# Galaxy glow
# ------------------------------------------------------------

final_halo = Circle(
    (0, 0),
    0.90,
    facecolor="#31566F",
    edgecolor="#52798D",
    linewidth=0.8,
    alpha=0.09,
    zorder=1
)

final_disk = plt.matplotlib.patches.Ellipse(
    (0, 0),
    1.55,
    0.48,
    angle=-8,
    facecolor="#70B6D0",
    edgecolor="#A7E5F7",
    linewidth=1.0,
    alpha=0.20,
    zorder=2
)

final_bulge = plt.matplotlib.patches.Ellipse(
    (0, 0),
    0.22,
    0.15,
    angle=-8,
    facecolor="#EAD69F",
    edgecolor="none",
    alpha=0.85,
    zorder=5
)

final_ax.add_patch(
    final_halo
)

final_ax.add_patch(
    final_disk
)

final_ax.add_patch(
    final_bulge
)


# ------------------------------------------------------------
# Synthetic catalogue points
# ------------------------------------------------------------

N_FINAL_DISK = 520

fr = rng.gamma(
    2.0,
    0.18,
    N_FINAL_DISK
)

fr = np.clip(
    fr,
    0.03,
    0.70
)

fp = rng.uniform(
    0,
    2*np.pi,
    N_FINAL_DISK
)

fx0 = (
    fr *
    np.cos(fp)
)

fy0 = (
    0.27 *
    fr *
    np.sin(fp)
)

ang = np.deg2rad(-8)

fx = (
    fx0 * np.cos(ang) -
    fy0 * np.sin(ang)
)

fy = (
    fx0 * np.sin(ang) +
    fy0 * np.cos(ang)
)

final_ax.scatter(
    fx,
    fy,
    s=rng.uniform(
        2,
        5,
        N_FINAL_DISK
    ),
    c="#C7ECF7",
    alpha=0.57,
    linewidths=0,
    zorder=6
)


N_FINAL_HALO = 230

fhr = (
    0.18 +
    0.75 *
    rng.random(N_FINAL_HALO)**0.55
)

fhp = rng.uniform(
    0,
    2*np.pi,
    N_FINAL_HALO
)

fhx = (
    fhr *
    np.cos(fhp)
)

fhy = (
    0.78 *
    fhr *
    np.sin(fhp)
)

final_ax.scatter(
    fhx,
    fhy,
    s=rng.uniform(
        2,
        4,
        N_FINAL_HALO
    ),
    c="#78AFC5",
    alpha=0.44,
    linewidths=0,
    zorder=4
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.815,
    "ONE HOMOGENISED 6D CATALOGUE",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_sub = fig.text(
    0.5, 0.772,
    "Different surveys transformed into a common phase-space dataset",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_input = fig.text(
    0.67, 0.655,
    "52+ MILLION",
    ha="center",
    va="center",
    color="#A7C5D2",
    fontsize=20,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_input_sub = fig.text(
    0.67, 0.615,
    "INPUT RECORDS",
    ha="center",
    va="center",
    color="#7E9BA8",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_arrow = fig.text(
    0.67, 0.555,
    "↓",
    ha="center",
    va="center",
    color="#79DDF8",
    fontsize=20,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_output = fig.text(
    0.67, 0.485,
    "32,552,876",
    ha="center",
    va="center",
    color="#C5F5FF",
    fontsize=27,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_output_sub = fig.text(
    0.67, 0.438,
    "UNIQUE STARS",
    ha="center",
    va="center",
    color="#A4CBD9",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_process = fig.text(
    0.67, 0.350,
    "CROSS-MATCH  →  CALIBRATE",
    ha="center",
    va="center",
    color="#B1E9F8",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_process2 = fig.text(
    0.67, 0.310,
    "DEDUPLICATE  →  COMBINE",
    ha="center",
    va="center",
    color="#B1E9F8",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_sixd = fig.text(
    0.67, 0.230,
    "X • Y • Z   +   VX • VY • VZ",
    ha="center",
    va="center",
    color="#D7F5FF",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_note = fig.text(
    0.5, 0.115,
    "THE ANIMATION ILLUSTRATES THE HOMOGENISATION WORKFLOW — MEASUREMENTS SHOWN ARE SCHEMATIC",
    ha="center",
    va="center",
    color="#7895A3",
    fontsize=8.5,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# Decorative only — not a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 4.5:

        phase_text.set_text(
            "MANY SURVEYS OBSERVE THE SAME GALAXY"
        )

        phase_sub.set_text(
            "Gaia astrometry is combined with major spectroscopic surveys"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "THE SAME STAR CAN APPEAR MORE THAN ONCE"
        )

        phase_sub.set_text(
            "Measurements come from different instruments and data releases"
        )

    else:

        phase_text.set_text(
            "BUILD ONE CONSISTENT CATALOGUE"
        )

        phase_sub.set_text(
            "Cross-match, calibrate, deduplicate and combine"
        )


    # ========================================================
    # SURVEYS APPEAR SEQUENTIALLY
    # ========================================================

    for i in range(
        len(survey_names)
    ):

        start = (
            0.4 +
            i * 0.48
        )

        p = smoothstep(
            (sec - start) / 0.65
        )

        survey_boxes[i].set_alpha(
            0.90 * p
        )

        survey_texts[i].set_alpha(
            p
        )

        stream_lines[i].set_alpha(
            0.60 * p
        )


    input_p = smoothstep(
        (sec - 2.6) / 1.0
    )

    input_counter.set_alpha(
        input_p
    )


    # ========================================================
    # MULTIPLE MEASUREMENTS
    # ========================================================

    measurement_p = smoothstep(
        (sec - 4.1) / 1.0
    )

    center_header.set_alpha(
        measurement_p
    )

    reference_glow.set_alpha(
        0.12 * measurement_p
    )

    reference_star.set_alpha(
        measurement_p
    )


    for i, point in enumerate(
        measurement_points
    ):

        pp = smoothstep(
            (
                sec -
                (4.3 + i * 0.18)
            ) /
            0.6
        )

        point.set_alpha(
            0.88 * pp
        )


    for i, txt in enumerate(
        rv_labels
    ):

        pp = smoothstep(
            (
                sec -
                (5.0 + i * 0.22)
            ) /
            0.6
        )

        txt.set_alpha(
            0.78 * pp
        )


    measurement_note.set_alpha(
        smoothstep(
            (sec - 5.5) / 0.8
        )
    )


    # ========================================================
    # PIPELINE
    # ========================================================

    pipe_p = smoothstep(
        (sec - 7.6) / 0.7
    )

    pipe_header.set_alpha(
        pipe_p
    )


    for i in range(
        len(pipeline_names)
    ):

        start = (
            8.0 +
            i * 1.05
        )

        pp = smoothstep(
            (sec - start) / 0.75
        )

        pipeline_boxes[i].set_alpha(
            0.92 * pp
        )

        pipeline_texts[i].set_alpha(
            pp
        )

        if i < len(pipeline_arrows):

            ap = smoothstep(
                (
                    sec -
                    (start + 0.55)
                ) /
                0.45
            )

            pipeline_arrows[i].set_alpha(
                ap
            )


    output_counter.set_alpha(
        smoothstep(
            (sec - 11.2) / 1.0
        )
    )


    # ========================================================
    # VISUAL DEDUPLICATION
    #
    # Duplicate measurements converge toward the reference star.
    # ========================================================

    dedup_p = smoothstep(
        (sec - 9.8) / 2.1
    )

    for i, point in enumerate(
        measurement_points
    ):

        dx, dy = offsets[i]

        x = (
            dx *
            (1.0 - dedup_p)
        )

        y = (
            0.36 +
            dy *
            (1.0 - dedup_p)
        )

        point.set_data(
            [x],
            [y]
        )

        point.set_alpha(
            0.88 *
            (
                1.0 -
                0.68 * dedup_p
            )
        )


    # RV values fade as catalogue consolidates
    for txt in rv_labels:

        txt.set_alpha(
            0.78 *
            (
                1.0 -
                0.70 * dedup_p
            )
        )


    # reference becomes brighter
    reference_glow.set_alpha(
        (
            0.12 +
            0.12 * dedup_p
        ) *
        measurement_p
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    survey_ax.set_alpha(
        scene1_alpha
    )

    center_ax.set_alpha(
        scene1_alpha
    )

    pipe_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )

    input_counter.set_alpha(
        input_p *
        scene1_alpha
    )

    output_counter.set_alpha(
        smoothstep(
            (sec - 11.2) / 1.0
        ) *
        scene1_alpha
    )


    # ========================================================
    # FINAL SCENE
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.98 * scene2
        )

        final_ax.set_visible(
            True
        )

        final_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_sub.set_alpha(
        scene2
    )

    final_input.set_alpha(
        scene2
    )

    final_input_sub.set_alpha(
        scene2
    )

    final_arrow.set_alpha(
        scene2
    )

    final_output.set_alpha(
        scene2
    )

    final_output_sub.set_alpha(
        scene2
    )

    final_process.set_alpha(
        scene2
    )

    final_process2.set_alpha(
        scene2
    )

    final_sixd.set_alpha(
        scene2
    )

    final_note.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Как из множества обзоров построить одну карту Млечного Пути

Современная карта Галактики создаётся не одним телескопом. Gaia чрезвычайно точно измеряет положения, параллаксы и собственные движения звёзд, а крупные спектроскопические программы добавляют лучевые скорости и другие параметры. В новой работе авторы объединили Gaia DR3 и 14 крупных спектроскопических обзоров. 2609.28652v1.pdf

Но простого объединения таблиц недостаточно. Одна и та же звезда может присутствовать сразу в нескольких обзорах, а измерения, полученные различными инструментами, имеют собственные систематические смещения и неопределённости. Поэтому авторы выполняют перекрёстное сопоставление источников, нормализуют ошибки, корректируют нулевые точки лучевых скоростей относительно Gaia и удаляют повторные наблюдения одних и тех же объектов.

Исходная сборка содержит более 52 миллионов записей. После калибровки и дедупликации они превращаются в единый однородный каталог из 32 552 876 уникальных звёзд с полными шестимерными фазовыми координатами. 2609.28652v1.pdf

Результат — не просто большая база данных. Homogenisation позволяет сравнивать звёзды, измеренные различными обзорами, в общей системе и использовать их как единый набор трассеров структуры и динамики Млечного Пути.

Показанные в анимации измерения одной звезды, значения лучевой скорости и процесс их сведения схематичны. Они иллюстрируют принцип построения каталога и не являются строками реального набора данных.

# How Multiple Surveys Become One Map of the Milky Way

A modern map of the Galaxy cannot be built with a single telescope. Gaia provides extremely precise positions, parallaxes, and proper motions, while large spectroscopic programmes contribute radial velocities and other stellar measurements. In the new study, the authors combine Gaia DR3 with 14 major spectroscopic surveys. 2609.28652v1.pdf

Simply concatenating those catalogues, however, would not produce a consistent Galactic map. The same star can appear in several surveys, while measurements obtained by different instruments carry their own systematic offsets and uncertainties. The authors therefore cross-match sources, normalise uncertainties, correct radial-velocity zero points relative to Gaia, and deduplicate repeated observations of the same objects.

The initial compilation contains more than 52 million records. After calibration and deduplication, these become a single homogenised catalogue of 32,552,876 unique stars with complete six-dimensional phase-space information. 2609.28652v1.pdf

The result is more than a very large database. Homogenisation allows stars observed by different surveys to be compared within a common framework and treated as a unified set of tracers of the structure and dynamics of the Milky Way.

The repeated stellar measurements, radial velocities, and consolidation process shown in the animation are schematic. They illustrate the catalogue-building procedure and are not individual records from the actual dataset.

---

# Part 3

In [ ]:
# ============================================================
# BEYOND GAIA — MAPPING THE DISTANT MILKY WAY
#
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4.5 s    Nearby stars: Gaia parallax
#   4.5–8 s    Parallax becomes harder with distance
#   8–13.2 s   Other distance tracers extend the map
#   13.2–14 s  Transition
#   14–20 s    Final distance ladder / uncertainty summary
#
# IMPORTANT
# The stars, distances, parallax angles and uncertainty graphics
# are SCHEMATIC. This is not a plot of individual catalogue stars.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse, Arc
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "beyond_gaia_distant_milky_way.webm"
else:
    OUTFILE = "beyond_gaia_distant_milky_way.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"

rng = np.random.default_rng(281)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "BEYOND GAIA: MAPPING THE DISTANT MILKY WAY",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Different distance tracers extend the Galactic map into the halo",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# LEFT — GALACTIC DISTANCE MAP
# ============================================================

ax = fig.add_axes(
    [0.045, 0.17, 0.65, 0.69],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -1.03,
    1.03
)

ax.set_ylim(
    -1.03,
    1.03
)

ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# GALACTIC STRUCTURE
# ============================================================

halo = Circle(
    (0, 0),
    0.94,
    facecolor="#31566F",
    edgecolor="#456C80",
    linewidth=0.8,
    alpha=0.035,
    zorder=1
)

disk = Ellipse(
    (0, 0),
    1.47,
    0.44,
    angle=-8,
    facecolor="#6EACC4",
    edgecolor="none",
    alpha=0.055,
    zorder=2
)

bulge = Ellipse(
    (0, 0),
    0.18,
    0.13,
    angle=-8,
    facecolor="#E8D49F",
    edgecolor="none",
    alpha=0.70,
    zorder=4
)

ax.add_patch(halo)
ax.add_patch(disk)
ax.add_patch(bulge)


# ============================================================
# SUN
# ============================================================

SUN_X = 0.27
SUN_Y = -0.03

sun_glow = Circle(
    (SUN_X, SUN_Y),
    0.042,
    facecolor="#FFE39B",
    edgecolor="none",
    alpha=0.17,
    zorder=30
)

sun = Circle(
    (SUN_X, SUN_Y),
    0.012,
    facecolor="#FFF1BC",
    edgecolor="white",
    linewidth=0.8,
    zorder=31
)

ax.add_patch(sun_glow)
ax.add_patch(sun)

sun_label = ax.text(
    SUN_X,
    SUN_Y - 0.075,
    "SUN",
    ha="center",
    va="center",
    color="#FFEFC0",
    fontsize=8,
    fontweight="bold",
    zorder=32
)


# ============================================================
# DISTANCE ZONES
# Schematic only.
# ============================================================

zone_near = Circle(
    (SUN_X, SUN_Y),
    0.17,
    facecolor="#83DFF5",
    edgecolor="#8EEAFF",
    linewidth=1.0,
    alpha=0.035,
    zorder=5
)

zone_mid = Circle(
    (SUN_X, SUN_Y),
    0.40,
    facecolor="none",
    edgecolor="#55899F",
    linewidth=0.9,
    linestyle="--",
    alpha=0.24,
    zorder=5
)

zone_far = Circle(
    (0, 0),
    0.91,
    facecolor="none",
    edgecolor="#466A7B",
    linewidth=0.9,
    linestyle="--",
    alpha=0.20,
    zorder=5
)

ax.add_patch(zone_near)
ax.add_patch(zone_mid)
ax.add_patch(zone_far)


# ============================================================
# NEARBY GAIA-LIKE STARS
# ============================================================

N_NEAR = 180

nr = (
    0.16 *
    np.sqrt(
        rng.random(N_NEAR)
    )
)

np_ = rng.uniform(
    0,
    2*np.pi,
    N_NEAR
)

near_x = (
    SUN_X +
    nr *
    np.cos(np_)
)

near_y = (
    SUN_Y +
    nr *
    np.sin(np_)
)

near_scatter = ax.scatter(
    near_x,
    near_y,
    s=rng.uniform(
        5,
        14,
        N_NEAR
    ),
    c="#D9F7FF",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ============================================================
# INTERMEDIATE DISK STARS
# ============================================================

N_MID = 450

mr = rng.gamma(
    2.0,
    0.18,
    N_MID
)

mr = np.clip(
    mr,
    0.08,
    0.67
)

mp = rng.uniform(
    0,
    2*np.pi,
    N_MID
)

mx0 = (
    mr *
    np.cos(mp)
)

my0 = (
    0.26 *
    mr *
    np.sin(mp)
)

rot = np.deg2rad(-8)

mid_x = (
    mx0 * np.cos(rot) -
    my0 * np.sin(rot)
)

mid_y = (
    mx0 * np.sin(rot) +
    my0 * np.cos(rot)
)

mid_scatter = ax.scatter(
    mid_x,
    mid_y,
    s=rng.uniform(
        2,
        6,
        N_MID
    ),
    c="#B4DDE9",
    alpha=0,
    linewidths=0,
    zorder=10
)


# ============================================================
# DISTANT HALO TRACERS
# ============================================================

N_FAR = 380

fr = (
    0.28 +
    0.64 *
    rng.random(N_FAR)**0.52
)

fp = rng.uniform(
    0,
    2*np.pi,
    N_FAR
)

far_x = (
    fr *
    np.cos(fp)
)

far_y = (
    0.80 *
    fr *
    np.sin(fp)
)

far_scatter = ax.scatter(
    far_x,
    far_y,
    s=rng.uniform(
        2,
        5,
        N_FAR
    ),
    c="#77ABC0",
    alpha=0,
    linewidths=0,
    zorder=8
)


# ============================================================
# SPECIAL DISTANCE TRACERS
# RR Lyrae / BHB / K giants — schematic categories
# ============================================================

tracer_idx = rng.choice(
    N_FAR,
    54,
    replace=False
)

tracer_x = far_x[tracer_idx]
tracer_y = far_y[tracer_idx]

types = np.arange(
    len(tracer_idx)
) % 3

rr_mask = (
    types == 0
)

bhb_mask = (
    types == 1
)

kg_mask = (
    types == 2
)


rr_scatter = ax.scatter(
    tracer_x[rr_mask],
    tracer_y[rr_mask],
    s=22,
    marker="o",
    facecolors="none",
    edgecolors="#B8F1FF",
    linewidths=1.0,
    alpha=0,
    zorder=20
)

bhb_scatter = ax.scatter(
    tracer_x[bhb_mask],
    tracer_y[bhb_mask],
    s=24,
    marker="^",
    c="#A3D7E8",
    linewidths=0,
    alpha=0,
    zorder=20
)

kg_scatter = ax.scatter(
    tracer_x[kg_mask],
    tracer_y[kg_mask],
    s=26,
    marker="D",
    facecolors="none",
    edgecolors="#79AFC4",
    linewidths=1.0,
    alpha=0,
    zorder=20
)


# ============================================================
# PARALLAX ILLUSTRATION
# ============================================================

# Two schematic Earth observing positions around Sun
earth1 = Circle(
    (SUN_X - 0.055, SUN_Y),
    0.007,
    facecolor="#8DDFF6",
    edgecolor="white",
    linewidth=0.5,
    alpha=0,
    zorder=40
)

earth2 = Circle(
    (SUN_X + 0.055, SUN_Y),
    0.007,
    facecolor="#8DDFF6",
    edgecolor="white",
    linewidth=0.5,
    alpha=0,
    zorder=40
)

ax.add_patch(earth1)
ax.add_patch(earth2)


target_x = SUN_X + 0.04
target_y = SUN_Y + 0.145

target_star = Circle(
    (target_x, target_y),
    0.012,
    facecolor="#E6FAFF",
    edgecolor="white",
    linewidth=0.6,
    alpha=0,
    zorder=42
)

ax.add_patch(target_star)


parallax_line1, = ax.plot(
    [SUN_X - 0.055, target_x],
    [SUN_Y, target_y],
    color="#82DDF6",
    lw=1.0,
    alpha=0,
    zorder=38
)

parallax_line2, = ax.plot(
    [SUN_X + 0.055, target_x],
    [SUN_Y, target_y],
    color="#82DDF6",
    lw=1.0,
    alpha=0,
    zorder=38
)


parallax_arc = Arc(
    (target_x, target_y),
    0.09,
    0.09,
    angle=0,
    theta1=245,
    theta2=295,
    color="#C5F5FF",
    lw=1.2,
    alpha=0,
    zorder=43
)

ax.add_patch(parallax_arc)


parallax_label = ax.text(
    target_x + 0.08,
    target_y + 0.015,
    "PARALLAX",
    ha="left",
    va="center",
    color="#C5F5FF",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=45
)


# ============================================================
# ZONE LABELS
# ============================================================

near_label = ax.text(
    SUN_X - 0.20,
    SUN_Y + 0.20,
    "GAIA PARALLAX",
    ha="center",
    va="center",
    color="#BFF4FF",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=45
)

near_sub = ax.text(
    SUN_X - 0.20,
    SUN_Y + 0.155,
    "precise nearby distances",
    ha="center",
    va="center",
    color="#84AAB9",
    fontsize=8,
    alpha=0,
    zorder=45
)


far_label = ax.text(
    -0.58,
    0.62,
    "DISTANT TRACERS",
    ha="center",
    va="center",
    color="#B0DFEE",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=45
)

far_sub = ax.text(
    -0.58,
    0.565,
    "RR Lyrae • BHB • K giants",
    ha="center",
    va="center",
    color="#779DAC",
    fontsize=8,
    alpha=0,
    zorder=45
)


# ============================================================
# RIGHT — UNCERTAINTY PANEL
# ============================================================

info_ax = fig.add_axes(
    [0.735, 0.20, 0.225, 0.57],
    zorder=15
)

info_ax.set_facecolor(
    "#050C13"
)

info_ax.set_xlim(
    0,
    1
)

info_ax.set_ylim(
    0,
    1
)

info_ax.set_xticks([])
info_ax.set_yticks([])

for spine in info_ax.spines.values():
    spine.set_color("#355466")


info_header = info_ax.text(
    0.5, 0.92,
    "DISTANCE",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


info_distance = info_ax.text(
    0.5, 0.80,
    "NEARBY",
    ha="center",
    va="center",
    color="#C3F4FF",
    fontsize=13,
    fontweight="bold"
)


# ============================================================
# UNCERTAINTY GRAPH
# ============================================================

gx0 = 0.18
gx1 = 0.84
gy0 = 0.26
gy1 = 0.68


info_ax.plot(
    [gx0, gx0],
    [gy0, gy1],
    color="#587888",
    lw=1.0
)

info_ax.plot(
    [gx0, gx1],
    [gy0, gy0],
    color="#587888",
    lw=1.0
)


info_ax.text(
    0.09,
    0.47,
    "RELATIVE\nUNCERTAINTY",
    ha="center",
    va="center",
    rotation=90,
    color="#7896A3",
    fontsize=7
)

info_ax.text(
    0.51,
    0.19,
    "DISTANCE",
    ha="center",
    va="center",
    color="#7896A3",
    fontsize=8
)


unc_line, = info_ax.plot(
    [],
    [],
    color="#9BE9FA",
    lw=2.2,
    zorder=10
)


unc_dot, = info_ax.plot(
    [],
    [],
    marker="o",
    markersize=6,
    color="#E5FBFF",
    zorder=12
)


unc_value = info_ax.text(
    0.5,
    0.08,
    "",
    ha="center",
    va="center",
    color="#BDEFFF",
    fontsize=10,
    fontweight="bold"
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.37, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.37, 0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL DISTANCE DIAGRAM
# ============================================================

final_ax = fig.add_axes(
    [0.06, 0.275, 0.47, 0.43],
    zorder=80
)

final_ax.set_facecolor(
    "#000207"
)

final_ax.set_xlim(
    0,
    250
)

final_ax.set_ylim(
    -0.5,
    2.5
)

final_ax.axis("off")
final_ax.set_visible(False)


# main line
final_ax.plot(
    [0, 250],
    [1, 1],
    color="#6D9CAE",
    lw=1.3,
    zorder=2
)


# distance ticks
for x in [
    0,
    15,
    50,
    100,
    250
]:

    final_ax.plot(
        [x, x],
        [0.91, 1.09],
        color="#A2CAD8",
        lw=1.0,
        zorder=3
    )

    final_ax.text(
        x,
        0.72,
        f"{x}",
        ha="center",
        va="center",
        color="#A6C0CB",
        fontsize=8,
        zorder=4
    )


final_ax.text(
    125,
    0.42,
    "DISTANCE FROM THE SUN / GALACTIC SCALE  [kpc]",
    ha="center",
    va="center",
    color="#738F9C",
    fontsize=8,
    zorder=4
)


# Gaia / nearby zone
final_ax.plot(
    [0, 15],
    [1.45, 1.45],
    color="#BDEFFF",
    lw=6,
    solid_capstyle="round",
    zorder=5
)

final_ax.text(
    7.5,
    1.76,
    "HIGH-PRECISION REGION",
    ha="center",
    va="center",
    color="#C8F4FF",
    fontsize=9,
    fontweight="bold",
    zorder=6
)

final_ax.text(
    7.5,
    1.58,
    "median distance uncertainty ≈ 2.7%",
    ha="center",
    va="center",
    color="#91B4C2",
    fontsize=8,
    zorder=6
)


# distant tracers
final_ax.plot(
    [15, 250],
    [1.45, 1.45],
    color="#5C91A8",
    lw=6,
    solid_capstyle="round",
    zorder=5
)

final_ax.text(
    132,
    1.76,
    "DISTANT GALACTIC TRACERS",
    ha="center",
    va="center",
    color="#A8D7E7",
    fontsize=9,
    fontweight="bold",
    zorder=6
)

final_ax.text(
    132,
    1.58,
    "uncertainties increase with distance",
    ha="center",
    va="center",
    color="#799AA8",
    fontsize=8,
    zorder=6
)


# far uncertainty marker
final_ax.text(
    224,
    2.12,
    "≈29%",
    ha="center",
    va="center",
    color="#BCEBFA",
    fontsize=13,
    fontweight="bold",
    zorder=8
)

final_ax.text(
    224,
    1.94,
    "median relative\nuncertainty at large distances",
    ha="center",
    va="center",
    color="#789BAA",
    fontsize=7.5,
    zorder=8
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5, 0.815,
    "EXTENDING THE GALACTIC DISTANCE SCALE",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_subheader = fig.text(
    0.5, 0.772,
    "No single distance technique maps the Milky Way from the Sun to its outer halo",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=10.5,
    alpha=0,
    zorder=100
)


final_right_1 = fig.text(
    0.74, 0.620,
    "NEARBY",
    ha="center",
    va="center",
    color="#C5F5FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_right_2 = fig.text(
    0.74, 0.575,
    "GAIA PARALLAX",
    ha="center",
    va="center",
    color="#A8D9E8",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_arrow = fig.text(
    0.74, 0.505,
    "↓",
    ha="center",
    va="center",
    color="#79DDF8",
    fontsize=20,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_right_3 = fig.text(
    0.74, 0.435,
    "DISTANT HALO",
    ha="center",
    va="center",
    color="#B4E6F5",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_right_4 = fig.text(
    0.74, 0.385,
    "RR LYRAE • BHB • K GIANTS",
    ha="center",
    va="center",
    color="#91B5C4",
    fontsize=9.5,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_250 = fig.text(
    0.74, 0.290,
    "MAP EXTENDS TO ~250 kpc",
    ha="center",
    va="center",
    color="#D5F4FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_note = fig.text(
    0.5, 0.115,
    "SCHEMATIC VISUALISATION — DISTANCE ZONES AND INDIVIDUAL STARS ARE NOT TO SCALE",
    ha="center",
    va="center",
    color="#7795A4",
    fontsize=8.5,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK / PROGRESS DIAL
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(dial_outer)
clock_ax.add_patch(dial_inner)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(dial_pin)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # NEARBY STARS
    # ========================================================

    near_p = smoothstep(
        sec / 1.6
    )

    near_scatter.set_alpha(
        0.88 * near_p
    )

    near_label.set_alpha(
        smoothstep(
            (sec - 0.8) / 0.8
        )
    )

    near_sub.set_alpha(
        smoothstep(
            (sec - 0.9) / 0.8
        )
    )


    # ========================================================
    # PARALLAX DEMONSTRATION
    # ========================================================

    parallax_p = smoothstep(
        (sec - 1.1) / 1.0
    )

    earth1.set_alpha(
        parallax_p
    )

    earth2.set_alpha(
        parallax_p
    )

    target_star.set_alpha(
        parallax_p
    )

    parallax_line1.set_alpha(
        0.75 * parallax_p
    )

    parallax_line2.set_alpha(
        0.75 * parallax_p
    )

    parallax_arc.set_alpha(
        parallax_p
    )

    parallax_label.set_alpha(
        parallax_p
    )


    # ========================================================
    # DISK EXTENSION
    # ========================================================

    mid_p = smoothstep(
        (sec - 4.2) / 2.0
    )

    mid_scatter.set_alpha(
        0.42 * mid_p
    )


    # ========================================================
    # HALO / OTHER TRACERS
    # ========================================================

    far_p = smoothstep(
        (sec - 7.6) / 2.0
    )

    far_scatter.set_alpha(
        0.33 * far_p
    )

    tracer_p = smoothstep(
        (sec - 8.4) / 1.5
    )

    rr_scatter.set_alpha(
        0.90 * tracer_p
    )

    bhb_scatter.set_alpha(
        0.90 * tracer_p
    )

    kg_scatter.set_alpha(
        0.90 * tracer_p
    )

    far_label.set_alpha(
        tracer_p
    )

    far_sub.set_alpha(
        tracer_p
    )


    # ========================================================
    # SUN PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 2.8
        )
    )

    sun_glow.set_alpha(
        0.11 +
        0.08 * pulse
    )


    # ========================================================
    # PHASE LABELS
    # ========================================================

    if sec < 4.5:

        phase_text.set_text(
            "PARALLAX MAPS THE NEARBY GALAXY"
        )

        phase_sub.set_text(
            "Apparent stellar displacement gives a geometric distance"
        )

        info_distance.set_text(
            "NEARBY"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "DISTANCE MAKES PARALLAX HARDER"
        )

        phase_sub.set_text(
            "The apparent displacement becomes progressively smaller"
        )

        info_distance.set_text(
            "FARTHER OUT"
        )

    else:

        phase_text.set_text(
            "OTHER TRACERS EXTEND THE MAP"
        )

        phase_sub.set_text(
            "Different stellar populations probe the distant disk and halo"
        )

        info_distance.set_text(
            "OUTER HALO"
        )


    # ========================================================
    # UNCERTAINTY GRAPH
    #
    # Conceptual trend anchored to the paper's reported
    # median values: ~2.7% within 15 kpc, rising to ~29%
    # at large distances.
    # ========================================================

    distance_p = smoothstep(
        sec / 12.5
    )

    xx = np.linspace(
        0,
        distance_p,
        120
    )

    # visual-only nonlinear curve
    yy = (
        0.07 +
        0.78 *
        xx**2.15
    )

    plot_x = (
        gx0 +
        (gx1 - gx0) *
        xx
    )

    plot_y = (
        gy0 +
        (gy1 - gy0) *
        yy
    )

    unc_line.set_data(
        plot_x,
        plot_y
    )

    if len(plot_x) > 0:

        unc_dot.set_data(
            [plot_x[-1]],
            [plot_y[-1]]
        )


    # displayed number interpolates only as a narrative indicator
    # between the two reported summary values
    if distance_p < 0.30:

        displayed_unc = (
            2.7
        )

    else:

        q = smoothstep(
            (distance_p - 0.30) / 0.70
        )

        displayed_unc = (
            2.7 +
            (29.0 - 2.7) * q
        )


    unc_value.set_text(
        f"MEDIAN Δd/d  ~{displayed_unc:.1f}%"
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL SCENE
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.98 * scene2
        )

        final_ax.set_visible(
            True
        )

        final_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_subheader.set_alpha(
        scene2
    )

    final_right_1.set_alpha(
        scene2
    )

    final_right_2.set_alpha(
        scene2
    )

    final_arrow.set_alpha(
        scene2
    )

    final_right_3.set_alpha(
        scene2
    )

    final_right_4.set_alpha(
        scene2
    )

    final_250.set_alpha(
        scene2
    )

    final_note.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# За пределами Gaia: как картографировать далёкий Млечный Путь

Один из главных способов измерять расстояния до ближайших звёзд — параллакс: при движении Земли вокруг Солнца близкая звезда немного смещается относительно далёкого фона. Gaia измеряет этот эффект с чрезвычайно высокой точностью.

Но чем дальше находится звезда, тем меньше её параллакс и тем труднее определить расстояние только геометрическим методом. Поэтому построение карты, простирающейся далеко в гало Млечного Пути, требует сочетания различных методов и звёздных трассеров.

В новом объединённом каталоге медианная относительная неопределённость расстояний составляет около 2,7% для звёзд в пределах 15 кпк, тогда как на больших расстояниях она возрастает примерно до 29%.

Для исследования далёких областей используются различные звёздные популяции, в том числе RR Lyrae, звёзды горизонтальной ветви BHB и K-гиганты. Они позволяют прослеживать структуру диска и гало на расстояниях, где одной параллактической информации уже недостаточно.

Комбинация этих методов помогает новому каталогу простираться примерно до 250 килопарсек, превращая карту окрестностей Солнца в карту значительной части галактического гало. 2609.28652v1.pdf

Звёзды, зоны расстояний, величина изображённого параллакса и график роста неопределённости в анимации схематические. Значения 2,7%, 29% и предельный масштаб около 250 кпк взяты из статьи.

# Beyond Gaia: Mapping the Distant Milky Way

One of the fundamental ways to measure distances to nearby stars is parallax: as Earth moves around the Sun, a nearby star appears to shift slightly against the distant background. Gaia measures this effect with extraordinary precision.

But the farther away a star lies, the smaller its parallax becomes, making distance increasingly difficult to determine from geometry alone. Mapping the Milky Way far into its halo therefore requires a combination of distance methods and different stellar tracers.

In the new homogenised catalogue, the median relative distance uncertainty is approximately 2.7% for stars within 15 kpc, increasing to roughly 29% at large distances.

Different stellar populations — including RR Lyrae stars, blue horizontal-branch stars, and K giants — can be used to probe more distant regions where parallax information alone is insufficient.

Combining these approaches allows the catalogue to extend to approximately 250 kiloparsecs, transforming a map of the Solar neighbourhood into a map reaching far into the Galactic halo. 2609.28652v1.pdf

The stars, distance zones, illustrated parallax angle, and uncertainty curve shown in the animation are schematic. The values of 2.7%, 29%, and the approximate 250 kpc spatial extent are taken from the study.

---

# Part 3

In [ ]:
# ============================================================
# THE OUTER HALO — 517,123 STELLAR TRACERS
#
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4 s      Familiar bright Galactic disk
#   4–8 s      Zoom outward into the stellar halo
#   8–13.2 s   Halo tracers / streams / satellites emerge
#   13.2–14 s  Transition
#   14–20 s    Final outer-halo summary
#
# IMPORTANT
# Stars, streams and satellites are SCHEMATIC.
# They are not individual catalogue sources.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "outer_halo_517123_tracers.webm"
else:
    OUTFILE = "outer_halo_517123_tracers.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"
rng = np.random.default_rng(517123)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# TITLES
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "THE OUTER HALO",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Following stellar tracers far beyond the Galactic disk",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# MAIN GALAXY PANEL
# ============================================================

ax = fig.add_axes(
    [0.045, 0.13, 0.72, 0.73],
    zorder=10
)

ax.set_facecolor(BG)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# GALAXY
# ============================================================

halo_glow = Circle(
    (0, 0),
    0.94,
    facecolor="#31566F",
    edgecolor="#42677A",
    linewidth=0.7,
    alpha=0.025,
    zorder=1
)

disk_outer = Ellipse(
    (0, 0),
    1.30,
    0.38,
    angle=-8,
    facecolor="#69AAC3",
    edgecolor="none",
    alpha=0.08,
    zorder=3
)

disk_inner = Ellipse(
    (0, 0),
    0.66,
    0.21,
    angle=-8,
    facecolor="#CBE5ED",
    edgecolor="none",
    alpha=0.12,
    zorder=4
)

bulge = Ellipse(
    (0, 0),
    0.17,
    0.12,
    angle=-8,
    facecolor="#EBD59E",
    edgecolor="none",
    alpha=0.82,
    zorder=5
)

ax.add_patch(halo_glow)
ax.add_patch(disk_outer)
ax.add_patch(disk_inner)
ax.add_patch(bulge)


# ============================================================
# DISK STARS
# ============================================================

N_DISK = 1000

r = rng.gamma(
    2.0,
    0.16,
    N_DISK
)

r = np.clip(
    r,
    0.02,
    0.62
)

phi = rng.uniform(
    0,
    2*np.pi,
    N_DISK
)

x0 = r * np.cos(phi)
y0 = 0.27 * r * np.sin(phi)

angle = np.deg2rad(-8)

disk_x = (
    x0 * np.cos(angle) -
    y0 * np.sin(angle)
)

disk_y = (
    x0 * np.sin(angle) +
    y0 * np.cos(angle)
)

disk_scatter = ax.scatter(
    disk_x,
    disk_y,
    s=rng.uniform(1.5, 6.0, N_DISK),
    c="#C7EAF4",
    alpha=0,
    linewidths=0,
    zorder=10
)


# ============================================================
# INNER HALO
# ============================================================

N_INNER = 500

ir = (
    0.18 +
    0.48 *
    rng.random(N_INNER)**0.60
)

ip = rng.uniform(
    0,
    2*np.pi,
    N_INNER
)

inner_x = ir * np.cos(ip)
inner_y = 0.80 * ir * np.sin(ip)

inner_scatter = ax.scatter(
    inner_x,
    inner_y,
    s=rng.uniform(1.8, 4.5, N_INNER),
    c="#80AFC2",
    alpha=0,
    linewidths=0,
    zorder=7
)


# ============================================================
# OUTER HALO
# ============================================================

N_OUTER = 700

orr = (
    0.48 +
    0.46 *
    rng.random(N_OUTER)**0.68
)

op = rng.uniform(
    0,
    2*np.pi,
    N_OUTER
)

outer_x = orr * np.cos(op)
outer_y = 0.84 * orr * np.sin(op)

outer_scatter = ax.scatter(
    outer_x,
    outer_y,
    s=rng.uniform(1.5, 4.2, N_OUTER),
    c="#628FA4",
    alpha=0,
    linewidths=0,
    zorder=6
)


# ============================================================
# METAL-POOR / FAINT TRACER SUBSET
# ============================================================

tracer_idx = rng.choice(
    N_OUTER,
    115,
    replace=False
)

tracer_scatter = ax.scatter(
    outer_x[tracer_idx],
    outer_y[tracer_idx],
    s=rng.uniform(7, 16, len(tracer_idx)),
    facecolors="none",
    edgecolors="#A9E7F7",
    linewidths=0.8,
    alpha=0,
    zorder=18
)


# ============================================================
# SCHEMATIC STREAM
# ============================================================

N_STREAM = 210

st = np.linspace(
    -1.45,
    1.20,
    N_STREAM
)

sr = (
    0.73 +
    0.045 *
    np.sin(
        2.6 * st
    )
)

stream_x = (
    sr *
    np.cos(st)
)

stream_y = (
    0.78 *
    sr *
    np.sin(st)
)

stream_x += rng.normal(
    0,
    0.008,
    N_STREAM
)

stream_y += rng.normal(
    0,
    0.008,
    N_STREAM
)

stream_scatter = ax.scatter(
    stream_x,
    stream_y,
    s=rng.uniform(2.5, 5.5, N_STREAM),
    c="#B1EAF8",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ============================================================
# SCHEMATIC SATELLITES
# ============================================================

sat_positions = [
    (-0.67, 0.48),
    (0.72, -0.42),
    (-0.80, -0.28),
    (0.58, 0.61)
]

satellites = []
satellite_glows = []

for x, y in sat_positions:

    glow = Circle(
        (x, y),
        0.035,
        facecolor="#8FD8EC",
        edgecolor="none",
        alpha=0,
        zorder=17
    )

    core = Circle(
        (x, y),
        0.009,
        facecolor="#D8F7FF",
        edgecolor="white",
        linewidth=0.5,
        alpha=0,
        zorder=18
    )

    ax.add_patch(glow)
    ax.add_patch(core)

    satellite_glows.append(glow)
    satellites.append(core)


# ============================================================
# SUN
# ============================================================

sun_glow = Circle(
    (0.24, -0.03),
    0.034,
    facecolor="#FFE19A",
    edgecolor="none",
    alpha=0.14,
    zorder=30
)

sun = Circle(
    (0.24, -0.03),
    0.010,
    facecolor="#FFF1BF",
    edgecolor="white",
    linewidth=0.7,
    zorder=31
)

ax.add_patch(sun_glow)
ax.add_patch(sun)


# ============================================================
# DISTANCE RINGS
# ============================================================

rings = []

for radius in [
    0.22,
    0.45,
    0.68,
    0.91
]:

    ring = Circle(
        (0, 0),
        radius,
        facecolor="none",
        edgecolor="#456C7D",
        linewidth=0.7,
        linestyle="--",
        alpha=0,
        zorder=2
    )

    ax.add_patch(ring)
    rings.append(ring)


# ============================================================
# SCALE LABEL
# ============================================================

scale_box = fig.text(
    0.865,
    0.79,
    "",
    ha="center",
    va="center",
    color="#DDF7FF",
    fontsize=12,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#07131E",
        edgecolor="#52788C",
        linewidth=1.0
    ),
    zorder=50
)

scale_patch = scale_box.get_bbox_patch()


# ============================================================
# RIGHT INFO PANEL
# ============================================================

info_ax = fig.add_axes(
    [0.785, 0.25, 0.175, 0.43],
    zorder=15
)

info_ax.set_facecolor("#050C13")
info_ax.set_xlim(0, 1)
info_ax.set_ylim(0, 1)
info_ax.set_xticks([])
info_ax.set_yticks([])

for spine in info_ax.spines.values():
    spine.set_color("#355466")


info_header = info_ax.text(
    0.5,
    0.90,
    "HALO TRACERS",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=11,
    fontweight="bold"
)


info_number = info_ax.text(
    0.5,
    0.72,
    "517,123",
    ha="center",
    va="center",
    color="#C6F5FF",
    fontsize=19,
    fontweight="bold",
    alpha=0
)


info_faint = info_ax.text(
    0.5,
    0.53,
    "MOSTLY FAINT",
    ha="center",
    va="center",
    color="#A9D9E9",
    fontsize=10,
    fontweight="bold",
    alpha=0
)

info_faint2 = info_ax.text(
    0.5,
    0.45,
    "G > 17",
    ha="center",
    va="center",
    color="#819FAC",
    fontsize=10,
    alpha=0
)


info_metal = info_ax.text(
    0.5,
    0.30,
    "METAL-POOR",
    ha="center",
    va="center",
    color="#A9D9E9",
    fontsize=10,
    fontweight="bold",
    alpha=0
)

info_metal2 = info_ax.text(
    0.5,
    0.22,
    "[Fe/H] < −1",
    ha="center",
    va="center",
    color="#819FAC",
    fontsize=10,
    alpha=0
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.40,
    0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.40,
    0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor("#000207")
result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL GALAXY
# ============================================================

final_ax = fig.add_axes(
    [0.055, 0.255, 0.39, 0.50],
    zorder=80
)

final_ax.set_facecolor("#000207")
final_ax.set_xlim(-1, 1)
final_ax.set_ylim(-1, 1)
final_ax.set_aspect("equal")
final_ax.axis("off")
final_ax.set_visible(False)


fg_halo = Circle(
    (0, 0),
    0.92,
    facecolor="#31566F",
    edgecolor="#527D93",
    linewidth=0.8,
    alpha=0.08,
    zorder=1
)

fg_disk = Ellipse(
    (0, 0),
    1.25,
    0.36,
    angle=-8,
    facecolor="#70B6D0",
    edgecolor="#A5E4F5",
    linewidth=0.8,
    alpha=0.18,
    zorder=3
)

fg_bulge = Ellipse(
    (0, 0),
    0.17,
    0.12,
    angle=-8,
    facecolor="#EBD69F",
    edgecolor="none",
    alpha=0.85,
    zorder=5
)

final_ax.add_patch(fg_halo)
final_ax.add_patch(fg_disk)
final_ax.add_patch(fg_bulge)


final_outer_idx = rng.choice(
    N_OUTER,
    430,
    replace=False
)

final_ax.scatter(
    outer_x[final_outer_idx],
    outer_y[final_outer_idx],
    s=4,
    c="#7FB2C7",
    alpha=0.50,
    linewidths=0,
    zorder=4
)

final_ax.scatter(
    stream_x,
    stream_y,
    s=4,
    c="#B9EFFB",
    alpha=0.72,
    linewidths=0,
    zorder=8
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5,
    0.815,
    "THE MILKY WAY EXTENDS FAR BEYOND ITS DISK",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_sub = fig.text(
    0.5,
    0.772,
    "Distant stellar tracers probe the Galaxy's outer halo",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_number = fig.text(
    0.69,
    0.640,
    "517,123",
    ha="center",
    va="center",
    color="#C5F5FF",
    fontsize=29,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_number_sub = fig.text(
    0.69,
    0.590,
    "HALO STARS",
    ha="center",
    va="center",
    color="#A7CEDC",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_types = fig.text(
    0.69,
    0.490,
    "RR LYRAE • BHB • K GIANTS",
    ha="center",
    va="center",
    color="#B5E8F7",
    fontsize=11,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_character = fig.text(
    0.69,
    0.405,
    "PREDOMINANTLY",
    ha="center",
    va="center",
    color="#799BAA",
    fontsize=9,
    alpha=0,
    zorder=100
)

final_character2 = fig.text(
    0.69,
    0.365,
    "FAINT  G > 17   •   METAL-POOR  [Fe/H] < −1",
    ha="center",
    va="center",
    color="#B0DCEA",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_250 = fig.text(
    0.69,
    0.270,
    "OUT TO ~250 kpc",
    ha="center",
    va="center",
    color="#D8F3FC",
    fontsize=15,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_note = fig.text(
    0.5,
    0.115,
    "SCHEMATIC VISUALISATION — STARS, STREAMS AND SATELLITES ARE NOT INDIVIDUAL CATALOGUE OBJECTS",
    ha="center",
    va="center",
    color="#7795A4",
    fontsize=8.3,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK
# Decorative only — not a physical timescale
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)
clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(dial_outer)
clock_ax.add_patch(dial_inner)


for i in range(12):

    a = 2 * np.pi * i / 12

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = dial_radius * 0.83

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = dial_radius * 0.56

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(dial_pin)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # DISK
    # ========================================================

    disk_p = smoothstep(
        sec / 1.5
    )

    disk_scatter.set_alpha(
        0.56 * disk_p
    )


    # ========================================================
    # INNER HALO
    # ========================================================

    inner_p = smoothstep(
        (sec - 3.2) / 2.0
    )

    inner_scatter.set_alpha(
        0.36 * inner_p
    )


    # ========================================================
    # OUTER HALO
    # ========================================================

    outer_p = smoothstep(
        (sec - 5.3) / 2.4
    )

    outer_scatter.set_alpha(
        0.34 * outer_p
    )

    halo_glow.set_alpha(
        0.025 +
        0.035 * outer_p
    )


    # ========================================================
    # DISTANCE RINGS
    # ========================================================

    for i, ring in enumerate(rings):

        rp = smoothstep(
            (
                sec -
                (2.0 + i * 0.8)
            ) /
            0.8
        )

        ring.set_alpha(
            0.26 * rp
        )


    # ========================================================
    # SPECIAL TRACERS
    # ========================================================

    tracer_p = smoothstep(
        (sec - 7.5) / 1.5
    )

    tracer_scatter.set_alpha(
        0.80 * tracer_p
    )

    info_number.set_alpha(
        tracer_p
    )

    info_faint.set_alpha(
        tracer_p
    )

    info_faint2.set_alpha(
        tracer_p
    )

    info_metal.set_alpha(
        tracer_p
    )

    info_metal2.set_alpha(
        tracer_p
    )


    # ========================================================
    # STREAM
    # ========================================================

    stream_p = smoothstep(
        (sec - 8.5) / 1.5
    )

    stream_scatter.set_alpha(
        0.70 * stream_p
    )


    # ========================================================
    # SATELLITES
    # ========================================================

    sat_p = smoothstep(
        (sec - 9.4) / 1.4
    )

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 * np.pi * sec / 2.5
        )
    )

    for glow, core in zip(
        satellite_glows,
        satellites
    ):

        glow.set_alpha(
            (0.08 + 0.07 * pulse) *
            sat_p
        )

        core.set_alpha(
            sat_p
        )


    # ========================================================
    # SUN
    # ========================================================

    sun_glow.set_alpha(
        0.10 +
        0.07 *
        (
            0.5 +
            0.5 *
            np.sin(
                2*np.pi*sec/3.0
            )
        )
    )


    # ========================================================
    # PHASE TEXT / SCALE
    # ========================================================

    if sec < 4.0:

        phase_text.set_text(
            "THE BRIGHT GALACTIC DISK"
        )

        phase_sub.set_text(
            "Most familiar stars belong to the central stellar system"
        )

        p = smoothstep(sec / 4.0)

        scale_box.set_text(
            f"SCALE  ~{5 + 20*p:.0f} kpc"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "ZOOM OUT INTO THE STELLAR HALO"
        )

        phase_sub.set_text(
            "The density falls — but the Galaxy does not end"
        )

        p = smoothstep(
            (sec - 4.0) / 4.0
        )

        scale_box.set_text(
            f"SCALE  ~{25 + 75*p:.0f} kpc"
        )

    else:

        phase_text.set_text(
            "DISTANT TRACERS REVEAL STRUCTURE"
        )

        phase_sub.set_text(
            "Halo stars, streams and satellites preserve the Galaxy's history"
        )

        p = smoothstep(
            (sec - 8.0) / 5.0
        )

        scale_box.set_text(
            f"SCALE  ~{100 + 150*p:.0f} kpc"
        )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = 1.0 - scene2


    ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )

    scale_box.set_alpha(
        scene1_alpha
    )

    scale_patch.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL SCENE
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(True)
        result_bg.set_alpha(
            0.98 * scene2
        )

        final_ax.set_visible(True)
        final_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(False)
        final_ax.set_visible(False)


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(scene2)
    final_sub.set_alpha(scene2)

    final_number.set_alpha(scene2)
    final_number_sub.set_alpha(scene2)

    final_types.set_alpha(scene2)

    final_character.set_alpha(scene2)
    final_character2.set_alpha(scene2)

    final_250.set_alpha(scene2)
    final_note.set_alpha(scene2)


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = 2 * np.pi * t

    hand_x = (
        hand_length *
        np.sin(dial_angle)
    )

    hand_y = (
        hand_length *
        np.cos(dial_angle)
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(f"Saved: {OUTFILE}")

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Внешнее гало Млечного Пути: 517 тысяч звёздных трассеров

Яркий звёздный диск — лишь центральная часть Млечного Пути. За его пределами находится огромное и гораздо более разреженное звёздное гало, простирающееся на десятки и сотни килопарсек.

Новый объединённый 6D-каталог содержит 517 123 звезды гало. Многие из них находятся в области, где возможности прямых параллактических измерений Gaia становятся ограниченными. Значительная часть этой выборки состоит из слабых звёзд с G > 17 и бедных металлами объектов с [Fe/H] < −1. В каталог входят такие важные дальние трассеры, как RR Lyrae, BHB-звёзды и K-гиганты. 2609.28652v1.pdf

Именно такие объекты позволяют прослеживать структуру Млечного Пути далеко за пределами его яркого диска — вплоть примерно до 250 килопарсек. 2609.28652v1.pdf

Внешнее гало особенно интересно потому, что оно хранит следы прошлого Галактики. Звёздные потоки, спутники и кинематически связанные группы могут быть остатками систем, которые когда-то были захвачены и разрушены гравитационным полем Млечного Пути. Поэтому карта положения и движения далёких звёзд становится инструментом галактической археологии.

Авторы проверяют возможности каталога на реальных структурах, включая звёздные скопления, карликовые галактики и поток Стрельца, показывая, что объединённый набор данных способен прослеживать динамически связанные системы в галактическом гало.

Положение отдельных звёзд, потоков и спутников в анимации схематическое и не воспроизводит реальные строки каталога. Число трассеров гало, их характерные свойства и масштаб карты соответствуют данным статьи.

# The Outer Halo: 517,123 Stellar Tracers

The bright stellar disk is only the central part of the Milky Way. Beyond it lies an enormous and much more diffuse stellar halo, extending for tens or even hundreds of kiloparsecs.

The new homogenised 6D catalogue contains 517,123 halo stars. Many occupy a regime where direct Gaia parallax measurements become increasingly limited. Much of this sample consists of faint stars with G > 17 and metal-poor objects with [Fe/H] < −1, including important distant tracers such as RR Lyrae stars, BHB stars, and K giants. 2609.28652v1.pdf

These populations allow astronomers to trace the Milky Way far beyond its bright disk, reaching distances of approximately 250 kiloparsecs. 2609.28652v1.pdf

The outer halo is particularly valuable because it preserves evidence of the Galaxy’s past. Stellar streams, satellites, and kinematically related groups can be remnants of systems that were captured and disrupted by the Milky Way. Mapping both the positions and motions of these distant stars therefore provides a powerful tool for Galactic archaeology.

The authors validate the catalogue using real Galactic structures, including stellar clusters, dwarf galaxies, and the Sagittarius stream, demonstrating its ability to trace dynamically related populations across the Galactic halo.

The individual stars, streams, and satellites shown in the animation are schematic and do not reproduce actual catalogue entries. The halo sample size, characteristic properties, and spatial extent correspond to values reported in the study.

---

# Part 4

In [ ]:
# ============================================================
# HOW FAR CAN WE TRUST THE MAP?
# Distance and radial-velocity uncertainties across the
# homogenised Milky Way 6D catalogue
#
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4 s      Nearby stars — compact uncertainties
#   4–8 s      Move outward — distance errors grow
#   8–13.2 s   Outer halo — useful, but less precise
#   13.2–14 s  Transition
#   14–20 s    Final uncertainty summary
#
# IMPORTANT
# Error ellipses and individual stars are SCHEMATIC.
# The numerical summary values are taken from the paper.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "milky_way_6d_uncertainties.webm"
else:
    OUTFILE = "milky_way_6d_uncertainties.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"
rng = np.random.default_rng(831)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# TITLES
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "HOW FAR CAN WE TRUST THE MAP?",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Uncertainty grows with distance — but the map still reaches the outer halo",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# MAIN PANEL
# ============================================================

ax = fig.add_axes(
    [0.055, 0.17, 0.68, 0.68],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -0.08,
    1.05
)

ax.set_ylim(
    -0.48,
    0.48
)

ax.axis("off")


# ============================================================
# DISTANCE AXIS
# ============================================================

axis_y = -0.32

ax.plot(
    [0.04, 0.96],
    [axis_y, axis_y],
    color="#547485",
    lw=1.2,
    zorder=2
)


distance_marks = [
    (0.06, "0"),
    (0.22, "15"),
    (0.40, "50"),
    (0.61, "100"),
    (0.94, "250")
]

for x, label in distance_marks:

    ax.plot(
        [x, x],
        [axis_y - 0.018, axis_y + 0.018],
        color="#8DB4C3",
        lw=1.0,
        zorder=3
    )

    ax.text(
        x,
        axis_y - 0.055,
        label,
        ha="center",
        va="center",
        color="#91AEBB",
        fontsize=8,
        zorder=4
    )


ax.text(
    0.50,
    axis_y - 0.105,
    "DISTANCE  [kpc]",
    ha="center",
    va="center",
    color="#718C99",
    fontsize=8,
    zorder=4
)


# ============================================================
# SUN
# ============================================================

sun_glow = Circle(
    (0.055, 0),
    0.037,
    facecolor="#FFE29A",
    edgecolor="none",
    alpha=0.16,
    zorder=20
)

sun = Circle(
    (0.055, 0),
    0.011,
    facecolor="#FFF2C0",
    edgecolor="white",
    linewidth=0.7,
    zorder=21
)

ax.add_patch(sun_glow)
ax.add_patch(sun)


ax.text(
    0.055,
    -0.07,
    "SUN",
    ha="center",
    va="center",
    color="#FFEFC0",
    fontsize=8,
    fontweight="bold",
    zorder=22
)


# ============================================================
# SYNTHETIC STARS ALONG DISTANCE
# ============================================================

N_STARS = 105

star_x = rng.uniform(
    0.10,
    0.96,
    N_STARS
)

star_y = rng.normal(
    0,
    0.135,
    N_STARS
)

star_y = np.clip(
    star_y,
    -0.25,
    0.25
)


# brighter / larger nearby
sizes = (
    3.0 +
    8.0 *
    (1.0 - star_x)**1.7
)


stars = ax.scatter(
    star_x,
    star_y,
    s=sizes,
    c="#C5ECF7",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ============================================================
# SELECTED STARS WITH UNCERTAINTY ELLIPSES
# ============================================================

selected = np.argsort(star_x)[::9]

error_ellipses = []

for idx in selected:

    x = star_x[idx]
    y = star_y[idx]

    # visual uncertainty increases with distance
    width = (
        0.010 +
        0.090 *
        x**2.4
    )

    height = (
        0.012 +
        0.022 *
        x
    )

    ellipse = Ellipse(
        (x, y),
        width,
        height,
        facecolor="none",
        edgecolor="#8DDDF3",
        linewidth=0.9,
        alpha=0,
        zorder=12
    )

    ax.add_patch(
        ellipse
    )

    error_ellipses.append(
        ellipse
    )


# ============================================================
# VELOCITY VECTORS
# ============================================================

vec_selected = np.argsort(star_x)[4::12]

velocity_arrows = []

for idx in vec_selected:

    x = star_x[idx]
    y = star_y[idx]

    angle = rng.uniform(
        -0.8,
        0.8
    )

    length = rng.uniform(
        0.025,
        0.050
    )

    dx = (
        length *
        np.cos(angle)
    )

    dy = (
        length *
        np.sin(angle)
    )

    arrow = ax.annotate(
        "",
        xy=(
            x + dx,
            y + dy
        ),
        xytext=(
            x,
            y
        ),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#B7EFFB",
            lw=1.0,
            shrinkA=0,
            shrinkB=0,
            alpha=0
        ),
        zorder=25
    )

    velocity_arrows.append(
        arrow
    )


# ============================================================
# MOVING RANGE MARKER
# ============================================================

range_line, = ax.plot(
    [0.06, 0.06],
    [-0.28, 0.31],
    color="#8EE9FF",
    lw=1.2,
    linestyle="--",
    alpha=0.7,
    zorder=30
)

range_glow = plt.Rectangle(
    (0.04, -0.28),
    0.02,
    0.59,
    facecolor="#63C7E5",
    edgecolor="none",
    alpha=0.035,
    zorder=1
)

ax.add_patch(range_glow)


range_label = ax.text(
    0.06,
    0.35,
    "",
    ha="center",
    va="center",
    color="#C5F5FF",
    fontsize=11,
    fontweight="bold",
    zorder=35
)


# ============================================================
# ZONE LABELS
# ============================================================

near_label = ax.text(
    0.16,
    0.28,
    "INNER / NEARBY SAMPLE",
    ha="center",
    va="center",
    color="#BDEFFF",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=40
)

far_label = ax.text(
    0.76,
    0.28,
    "DISTANT HALO",
    ha="center",
    va="center",
    color="#A1CDDC",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=40
)


# ============================================================
# RIGHT INFORMATION PANEL
# ============================================================

info_ax = fig.add_axes(
    [0.765, 0.20, 0.195, 0.57],
    zorder=15
)

info_ax.set_facecolor(
    "#050C13"
)

info_ax.set_xlim(
    0,
    1
)

info_ax.set_ylim(
    0,
    1
)

info_ax.set_xticks([])
info_ax.set_yticks([])

for spine in info_ax.spines.values():
    spine.set_color("#355466")


info_title = info_ax.text(
    0.5,
    0.91,
    "MEDIAN UNCERTAINTY",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=10.5,
    fontweight="bold"
)


info_distance_title = info_ax.text(
    0.5,
    0.75,
    "DISTANCE",
    ha="center",
    va="center",
    color="#8EAEBB",
    fontsize=9,
    fontweight="bold"
)

info_distance = info_ax.text(
    0.5,
    0.64,
    "≈ 2.7%",
    ha="center",
    va="center",
    color="#C7F5FF",
    fontsize=22,
    fontweight="bold"
)

info_distance_sub = info_ax.text(
    0.5,
    0.54,
    "d ≤ 15 kpc",
    ha="center",
    va="center",
    color="#8FAEBA",
    fontsize=9
)


divider = info_ax.plot(
    [0.20, 0.80],
    [0.43, 0.43],
    color="#355466",
    lw=0.8
)[0]


info_rv_title = info_ax.text(
    0.5,
    0.33,
    "RADIAL VELOCITY",
    ha="center",
    va="center",
    color="#8EAEBB",
    fontsize=9,
    fontweight="bold"
)

info_rv = info_ax.text(
    0.5,
    0.22,
    "≈ 8.31%",
    ha="center",
    va="center",
    color="#B9EAF7",
    fontsize=19,
    fontweight="bold"
)

info_rv_sub = info_ax.text(
    0.5,
    0.12,
    "catalogue-wide median",
    ha="center",
    va="center",
    color="#7896A3",
    fontsize=8
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.39,
    0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.39,
    0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL UNCERTAINTY GRAPH
# ============================================================

final_ax = fig.add_axes(
    [0.075, 0.285, 0.43, 0.40],
    zorder=80
)

final_ax.set_facecolor(
    "#000207"
)

final_ax.set_xlim(
    0,
    250
)

final_ax.set_ylim(
    0,
    35
)

final_ax.set_visible(False)


for spine in [
    "top",
    "right"
]:
    final_ax.spines[spine].set_visible(False)

for spine in [
    "bottom",
    "left"
]:
    final_ax.spines[spine].set_color(
        "#526F7C"
    )


final_ax.tick_params(
    colors="#8DA6B1",
    labelsize=8
)

final_ax.set_xlabel(
    "DISTANCE [kpc]",
    color="#819AA6",
    fontsize=8
)

final_ax.set_ylabel(
    "RELATIVE DISTANCE UNCERTAINTY [%]",
    color="#819AA6",
    fontsize=8
)


# conceptual curve anchored at summary values
xx = np.linspace(
    0,
    250,
    300
)

yy = (
    2.7 +
    26.3 *
    (
        np.maximum(
            xx - 15,
            0
        ) /
        235
    )**0.70
)

final_ax.plot(
    xx,
    yy,
    color="#9CE9FA",
    lw=2.2
)


final_ax.scatter(
    [15],
    [2.7],
    s=42,
    c="#DDF9FF",
    zorder=10
)

final_ax.scatter(
    [250],
    [29.0],
    s=42,
    c="#DDF9FF",
    zorder=10
)


final_ax.text(
    15,
    5.2,
    "≈2.7%",
    ha="center",
    va="center",
    color="#D7F7FF",
    fontsize=9,
    fontweight="bold"
)

final_ax.text(
    225,
    30.5,
    "≈29%",
    ha="center",
    va="center",
    color="#D7F7FF",
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5,
    0.815,
    "A LARGE MAP IS NOT A PERFECT MAP",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_sub = fig.text(
    0.5,
    0.772,
    "The catalogue preserves uncertainties as the Galactic volume expands",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_near = fig.text(
    0.70,
    0.630,
    "≈ 2.7%",
    ha="center",
    va="center",
    color="#C6F5FF",
    fontsize=26,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_near_sub = fig.text(
    0.70,
    0.580,
    "MEDIAN DISTANCE UNCERTAINTY\nWITHIN 15 kpc",
    ha="center",
    va="center",
    color="#91B5C4",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_far = fig.text(
    0.70,
    0.455,
    "≈ 29%",
    ha="center",
    va="center",
    color="#B7E9F7",
    fontsize=24,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_far_sub = fig.text(
    0.70,
    0.405,
    "MEDIAN DISTANCE UNCERTAINTY\n15–250 kpc",
    ha="center",
    va="center",
    color="#829FAB",
    fontsize=9,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_rv = fig.text(
    0.70,
    0.295,
    "≈ 8.31%",
    ha="center",
    va="center",
    color="#BDEFFF",
    fontsize=22,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_rv_sub = fig.text(
    0.70,
    0.250,
    "MEDIAN FRACTIONAL\nRADIAL-VELOCITY UNCERTAINTY",
    ha="center",
    va="center",
    color="#829FAB",
    fontsize=8.5,
    fontweight="bold",
    alpha=0,
    zorder=100
)


final_note = fig.text(
    0.5,
    0.115,
    "ERROR ELLIPSES AND CURVE SHAPE ARE SCHEMATIC — SUMMARY VALUES ARE FROM THE STUDY",
    ha="center",
    va="center",
    color="#7795A4",
    fontsize=8.5,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)

clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # STARS
    # ========================================================

    star_p = smoothstep(
        sec / 1.5
    )

    stars.set_alpha(
        0.78 * star_p
    )


    # ========================================================
    # MOVING DISTANCE FRONT
    # ========================================================

    distance_p = smoothstep(
        sec / 12.8
    )

    front_x = (
        0.06 +
        0.88 * distance_p
    )

    range_line.set_data(
        [front_x, front_x],
        [-0.28, 0.31]
    )

    range_glow.set_width(
        max(front_x - 0.04, 0.001)
    )


    # narrative distance
    distance_kpc = (
        250 *
        distance_p
    )

    range_label.set_position(
        (
            min(
                front_x,
                0.87
            ),
            0.35
        )
    )

    range_label.set_text(
        f"~{distance_kpc:.0f} kpc"
    )


    # ========================================================
    # ERROR ELLIPSES
    # ========================================================

    for ellipse, idx in zip(
        error_ellipses,
        selected
    ):

        appear = smoothstep(
            (
                front_x -
                star_x[idx] +
                0.04
            ) /
            0.08
        )

        ellipse.set_alpha(
            0.68 *
            appear *
            star_p
        )


    # ========================================================
    # VELOCITY VECTORS
    # ========================================================

    velocity_p = smoothstep(
        (sec - 5.2) / 1.5
    )

    for arrow in velocity_arrows:

        arrow.arrow_patch.set_alpha(
            0.75 *
            velocity_p
        )


    # ========================================================
    # LABELS
    # ========================================================

    near_label.set_alpha(
        smoothstep(
            (sec - 0.8) / 0.8
        ) *
        (
            1 -
            0.7 *
            smoothstep(
                (sec - 6.0) / 2.0
            )
        )
    )

    far_label.set_alpha(
        smoothstep(
            (sec - 7.2) / 1.3
        )
    )


    # ========================================================
    # INFORMATION PANEL
    # ========================================================

    if distance_kpc <= 15:

        current_unc = 2.7

        info_distance_sub.set_text(
            "d ≤ 15 kpc"
        )

    else:

        q = smoothstep(
            (distance_kpc - 15) /
            235
        )

        current_unc = (
            2.7 +
            (29.0 - 2.7) *
            q
        )

        info_distance_sub.set_text(
            "increasing with distance"
        )


    info_distance.set_text(
        f"≈ {current_unc:.1f}%"
    )


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 4.0:

        phase_text.set_text(
            "NEARBY STARS ARE MEASURED PRECISELY"
        )

        phase_sub.set_text(
            "Within 15 kpc, median fractional distance uncertainty is only a few percent"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "UNCERTAINTY GROWS WITH DISTANCE"
        )

        phase_sub.set_text(
            "The farther the tracer, the less precisely its distance is known"
        )

    else:

        phase_text.set_text(
            "THE OUTER HALO IS STILL ACCESSIBLE"
        )

        phase_sub.set_text(
            "Large uncertainties do not make distant tracers scientifically useless"
        )


    # ========================================================
    # SUN PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec /
            2.8
        )
    )

    sun_glow.set_alpha(
        0.10 +
        0.08 *
        pulse
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.98 *
            scene2
        )

        final_ax.set_visible(
            True
        )

        final_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_ax.set_visible(
            False
        )


    final_header.set_alpha(
        scene2
    )

    final_sub.set_alpha(
        scene2
    )

    final_near.set_alpha(
        scene2
    )

    final_near_sub.set_alpha(
        scene2
    )

    final_far.set_alpha(
        scene2
    )

    final_far_sub.set_alpha(
        scene2
    )

    final_rv.set_alpha(
        scene2
    )

    final_rv_sub.set_alpha(
        scene2
    )

    final_note.set_alpha(
        scene2
    )


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = (
        2 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Насколько точна шестимерная карта Млечного Пути?

Размер каталога — это только часть задачи. Для динамической карты Галактики важно знать не только положение и скорость каждой звезды, но и неопределённость этих измерений.

В новом 6D-каталоге точность расстояний существенно зависит от удалённости звезды. Для объектов в пределах 15 килопарсек медианная относительная неопределённость расстояния составляет около 2,7%. Для гораздо более удалённых объектов, находящихся в диапазоне 15–250 кпк, она возрастает примерно до 29%.

Скорости тоже имеют свои ошибки. Для лучевой скорости авторы получают медианную относительную неопределённость около 8,31% по каталогу в целом.

Рост неопределённости не означает, что далёкие звёзды бесполезны. Напротив, даже менее точные измерения большого числа трассеров позволяют исследовать структуру внешнего гало, искать звёздные потоки и изучать динамически связанные популяции. Критически важно лишь учитывать ошибки при интерпретации шестимерной карты.

Именно поэтому каталог сохраняет не только объединённые координаты и скорости, но также их неопределённости и информацию о происхождении измерений. Это позволяет использовать огромный набор данных статистически, не создавая иллюзии одинаковой точности для ближайших и самых далёких объектов.

Положение звёзд, эллипсы ошибок и форма кривой роста неопределённости в анимации схематические. Численные значения 2,7%, 29% и 8,31% соответствуют приведённым в статье сводным характеристикам.

# How Far Can We Trust the Map?

Catalogue size is only part of the problem. A dynamical map of the Galaxy must describe not only the position and velocity of each star, but also the uncertainty associated with those measurements.

In the new 6D catalogue, distance precision depends strongly on how far away a star lies. For objects within 15 kiloparsecs, the median fractional distance uncertainty is approximately 2.7%. For much more distant objects spanning 15–250 kpc, it increases to roughly 29%.

Velocities also carry uncertainties. The catalogue-wide median fractional uncertainty in radial velocity is approximately 8.31%.

Increasing uncertainty does not make distant stars useless. Even less precise measurements of large tracer populations can reveal the structure of the outer halo, identify stellar streams, and constrain dynamically related populations. The essential requirement is to propagate those uncertainties when interpreting the six-dimensional map.

The catalogue therefore retains not only combined positions and velocities, but also their uncertainties and measurement provenance. This allows the enormous dataset to be analysed statistically without assuming that nearby and extremely distant stars are measured with equal precision.

The stellar positions, error ellipses, and shape of the uncertainty curve shown in the animation are schematic. The numerical summary values of 2.7%, 29%, and 8.31% correspond to those reported in the study.

---

# Part 5

In [ ]:
# ============================================================
# FROM A STAR MAP TO PHASE SPACE
#
# Inspired by:
# "Mapping the Milky Way in Six Dimensions:
#  A contiguous, homogenised, phase-space catalogue
#  of Gaia DR3 tracers up to 250 kpc"
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–4 s      3D positions: stars are points
#   4–8 s      Add velocities: VX, VY, VZ
#   8–13.2 s   Coherent motions reveal populations / stream
#   13.2–14 s  Transition
#   14–20 s    Final 3D vs 6D card
#
# IMPORTANT:
# Stellar positions, velocity vectors and stream geometry
# are SCHEMATIC. They illustrate the concept of phase space.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "from_star_map_to_phase_space.webm"
else:
    OUTFILE = "from_star_map_to_phase_space.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"
rng = np.random.default_rng(6006)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# TITLES
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "FROM A STAR MAP TO PHASE SPACE",
    ha="center",
    va="center",
    color="white",
    fontsize=21,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.916,
    "Position tells us where a star is — velocity tells us how it moves",
    ha="center",
    va="center",
    color="#B7D6E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# MAIN GALAXY PANEL
# ============================================================

ax = fig.add_axes(
    [0.045, 0.15, 0.70, 0.70],
    zorder=10
)

ax.set_facecolor(BG)
ax.set_xlim(-1.02, 1.02)
ax.set_ylim(-0.78, 0.78)
ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# GALAXY BACKGROUND
# ============================================================

halo = Ellipse(
    (0, 0),
    1.85,
    1.35,
    facecolor="#31566F",
    edgecolor="#42677A",
    linewidth=0.7,
    alpha=0.035,
    zorder=1
)

disk = Ellipse(
    (0, 0),
    1.40,
    0.45,
    angle=-8,
    facecolor="#69AAC3",
    edgecolor="none",
    alpha=0.065,
    zorder=2
)

bulge = Ellipse(
    (0, 0),
    0.18,
    0.13,
    angle=-8,
    facecolor="#EBD59E",
    edgecolor="none",
    alpha=0.80,
    zorder=4
)

ax.add_patch(halo)
ax.add_patch(disk)
ax.add_patch(bulge)


# ============================================================
# DISK POPULATION
# ============================================================

N_DISK = 430

r = rng.gamma(
    2.0,
    0.17,
    N_DISK
)

r = np.clip(
    r,
    0.06,
    0.67
)

phi = rng.uniform(
    0,
    2*np.pi,
    N_DISK
)

x0 = r * np.cos(phi)
y0 = 0.28 * r * np.sin(phi)

rot = np.deg2rad(-8)

disk_x = (
    x0 * np.cos(rot) -
    y0 * np.sin(rot)
)

disk_y = (
    x0 * np.sin(rot) +
    y0 * np.cos(rot)
)

disk_points = ax.scatter(
    disk_x,
    disk_y,
    s=rng.uniform(2.5, 7, N_DISK),
    c="#C5EAF4",
    alpha=0,
    linewidths=0,
    zorder=10
)


# ============================================================
# HALO POPULATION
# ============================================================

N_HALO = 230

hr = (
    0.30 +
    0.60 *
    rng.random(N_HALO)**0.65
)

hp = rng.uniform(
    0,
    2*np.pi,
    N_HALO
)

halo_x = hr * np.cos(hp)
halo_y = 0.72 * hr * np.sin(hp)

halo_points = ax.scatter(
    halo_x,
    halo_y,
    s=rng.uniform(2.5, 6, N_HALO),
    c="#7FAABC",
    alpha=0,
    linewidths=0,
    zorder=8
)


# ============================================================
# STREAM
# ============================================================

N_STREAM = 75

st = np.linspace(
    -1.25,
    0.95,
    N_STREAM
)

sr = (
    0.72 +
    0.025 *
    np.sin(3.0 * st)
)

stream_x = sr * np.cos(st)
stream_y = 0.69 * sr * np.sin(st)

stream_x += rng.normal(
    0,
    0.006,
    N_STREAM
)

stream_y += rng.normal(
    0,
    0.006,
    N_STREAM
)

stream_points = ax.scatter(
    stream_x,
    stream_y,
    s=7,
    c="#C4F3FF",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ============================================================
# SUN
# ============================================================

sun_glow = Circle(
    (0.25, -0.03),
    0.037,
    facecolor="#FFE29A",
    edgecolor="none",
    alpha=0.15,
    zorder=25
)

sun = Circle(
    (0.25, -0.03),
    0.011,
    facecolor="#FFF1BE",
    edgecolor="white",
    linewidth=0.7,
    zorder=26
)

ax.add_patch(sun_glow)
ax.add_patch(sun)

ax.text(
    0.25,
    -0.09,
    "SUN",
    ha="center",
    va="center",
    color="#FFEFC0",
    fontsize=8,
    fontweight="bold",
    zorder=27
)


# ============================================================
# VELOCITY VECTORS — DISK
# ============================================================

disk_vector_idx = rng.choice(
    N_DISK,
    44,
    replace=False
)

disk_arrows = []

for idx in disk_vector_idx:

    x = disk_x[idx]
    y = disk_y[idx]

    # tangential motion around Galactic centre
    rr = np.hypot(x, y) + 1e-6

    dx = (
        -y / rr *
        0.060
    )

    dy = (
        x / rr *
        0.060
    )

    arrow = ax.annotate(
        "",
        xy=(x + dx, y + dy),
        xytext=(x, y),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#9CE9FA",
            lw=1.0,
            shrinkA=0,
            shrinkB=0,
            alpha=0
        ),
        zorder=30
    )

    disk_arrows.append(arrow)


# ============================================================
# VELOCITY VECTORS — HALO
# ============================================================

halo_vector_idx = rng.choice(
    N_HALO,
    28,
    replace=False
)

halo_arrows = []

for idx in halo_vector_idx:

    x = halo_x[idx]
    y = halo_y[idx]

    a = rng.uniform(
        0,
        2*np.pi
    )

    length = rng.uniform(
        0.035,
        0.075
    )

    dx = length * np.cos(a)
    dy = length * np.sin(a)

    arrow = ax.annotate(
        "",
        xy=(x + dx, y + dy),
        xytext=(x, y),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#789EAE",
            lw=0.9,
            shrinkA=0,
            shrinkB=0,
            alpha=0
        ),
        zorder=25
    )

    halo_arrows.append(arrow)


# ============================================================
# VELOCITY VECTORS — STREAM
# ============================================================

stream_vector_idx = np.arange(
    3,
    N_STREAM,
    7
)

stream_arrows = []

for idx in stream_vector_idx:

    x = stream_x[idx]
    y = stream_y[idx]

    if idx < N_STREAM - 1:

        dx = (
            stream_x[idx + 1] -
            stream_x[idx]
        )

        dy = (
            stream_y[idx + 1] -
            stream_y[idx]
        )

    else:

        dx = (
            stream_x[idx] -
            stream_x[idx - 1]
        )

        dy = (
            stream_y[idx] -
            stream_y[idx - 1]
        )

    norm = (
        np.hypot(dx, dy) +
        1e-8
    )

    dx = (
        dx / norm *
        0.070
    )

    dy = (
        dy / norm *
        0.070
    )

    arrow = ax.annotate(
        "",
        xy=(x + dx, y + dy),
        xytext=(x, y),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#D8FAFF",
            lw=1.2,
            shrinkA=0,
            shrinkB=0,
            alpha=0
        ),
        zorder=35
    )

    stream_arrows.append(arrow)


# ============================================================
# STREAM HIGHLIGHT
# ============================================================

stream_label = ax.text(
    -0.58,
    0.54,
    "COHERENT MOTION",
    ha="center",
    va="center",
    color="#D1F7FF",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=50
)

stream_sub = ax.text(
    -0.58,
    0.49,
    "stars move together through phase space",
    ha="center",
    va="center",
    color="#87AAB8",
    fontsize=8,
    alpha=0,
    zorder=50
)


# ============================================================
# RIGHT PANEL
# ============================================================

info_ax = fig.add_axes(
    [0.775, 0.21, 0.19, 0.56],
    zorder=15
)

info_ax.set_facecolor(
    "#050C13"
)

info_ax.set_xlim(0, 1)
info_ax.set_ylim(0, 1)

info_ax.set_xticks([])
info_ax.set_yticks([])

for spine in info_ax.spines.values():
    spine.set_color("#355466")


info_title = info_ax.text(
    0.5,
    0.91,
    "PHASE SPACE",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


# ============================================================
# POSITION
# ============================================================

position_label = info_ax.text(
    0.5,
    0.78,
    "POSITION",
    ha="center",
    va="center",
    color="#8DAAB7",
    fontsize=9,
    fontweight="bold"
)

position_values = info_ax.text(
    0.5,
    0.68,
    "X   Y   Z",
    ha="center",
    va="center",
    color="#D5F5FD",
    fontsize=17,
    fontweight="bold"
)


plus_text = info_ax.text(
    0.5,
    0.53,
    "+",
    ha="center",
    va="center",
    color="#75D9F4",
    fontsize=22,
    fontweight="bold",
    alpha=0
)


# ============================================================
# VELOCITY
# ============================================================

velocity_label = info_ax.text(
    0.5,
    0.40,
    "VELOCITY",
    ha="center",
    va="center",
    color="#8DAAB7",
    fontsize=9,
    fontweight="bold",
    alpha=0
)

velocity_values = info_ax.text(
    0.5,
    0.30,
    "VX  VY  VZ",
    ha="center",
    va="center",
    color="#BCEFFC",
    fontsize=15,
    fontweight="bold",
    alpha=0
)


sixd_text = info_ax.text(
    0.5,
    0.13,
    "6D",
    ha="center",
    va="center",
    color="#E3FAFF",
    fontsize=25,
    fontweight="bold",
    alpha=0
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.40,
    0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.40,
    0.817,
    "",
    ha="center",
    va="center",
    color="#AFCBDC",
    fontsize=10,
    zorder=50
)


# ============================================================
# FINAL BACKGROUND
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor("#000207")
result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL LEFT — 3D
# ============================================================

final_left = fig.add_axes(
    [0.075, 0.265, 0.36, 0.43],
    zorder=80
)

final_left.set_facecolor("#000207")
final_left.set_xlim(-1, 1)
final_left.set_ylim(-0.7, 0.7)
final_left.axis("off")
final_left.set_visible(False)


final_left.add_patch(
    Ellipse(
        (0, 0),
        1.65,
        0.52,
        angle=-8,
        facecolor="#5793AA",
        edgecolor="none",
        alpha=0.08
    )
)

fl_idx = rng.choice(
    N_DISK,
    180,
    replace=False
)

final_left.scatter(
    disk_x[fl_idx],
    disk_y[fl_idx],
    s=5,
    c="#C5EAF4",
    alpha=0.58,
    linewidths=0
)

fh_idx = rng.choice(
    N_HALO,
    80,
    replace=False
)

final_left.scatter(
    halo_x[fh_idx],
    halo_y[fh_idx],
    s=4,
    c="#779DAC",
    alpha=0.45,
    linewidths=0
)

final_left.text(
    0,
    0.56,
    "3D POSITION",
    ha="center",
    va="center",
    color="#D8F6FF",
    fontsize=13,
    fontweight="bold"
)

final_left.text(
    0,
    -0.57,
    "WHERE ARE THE STARS?",
    ha="center",
    va="center",
    color="#829EAA",
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# FINAL RIGHT — 6D
# ============================================================

final_right = fig.add_axes(
    [0.565, 0.265, 0.36, 0.43],
    zorder=80
)

final_right.set_facecolor("#000207")
final_right.set_xlim(-1, 1)
final_right.set_ylim(-0.7, 0.7)
final_right.axis("off")
final_right.set_visible(False)


final_right.add_patch(
    Ellipse(
        (0, 0),
        1.65,
        0.52,
        angle=-8,
        facecolor="#5793AA",
        edgecolor="none",
        alpha=0.08
    )
)

final_right.scatter(
    disk_x[fl_idx],
    disk_y[fl_idx],
    s=5,
    c="#C5EAF4",
    alpha=0.58,
    linewidths=0
)

final_right.scatter(
    halo_x[fh_idx],
    halo_y[fh_idx],
    s=4,
    c="#779DAC",
    alpha=0.45,
    linewidths=0
)


# schematic velocity vectors in final panel
final_vec_idx = fl_idx[:30]

for idx in final_vec_idx:

    x = disk_x[idx]
    y = disk_y[idx]

    rr = np.hypot(x, y) + 1e-8

    dx = -y / rr * 0.055
    dy = x / rr * 0.055

    final_right.annotate(
        "",
        xy=(x + dx, y + dy),
        xytext=(x, y),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#A4EAF9",
            lw=0.8,
            shrinkA=0,
            shrinkB=0,
            alpha=0.75
        )
    )


final_right.text(
    0,
    0.56,
    "6D PHASE SPACE",
    ha="center",
    va="center",
    color="#D8F6FF",
    fontsize=13,
    fontweight="bold"
)

final_right.text(
    0,
    -0.57,
    "WHERE ARE THEY GOING?",
    ha="center",
    va="center",
    color="#A6D5E5",
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# FINAL TEXT
# ============================================================

final_header = fig.text(
    0.5,
    0.815,
    "POSITION IS ONLY HALF THE STORY",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_subheader = fig.text(
    0.5,
    0.770,
    "Velocity turns a static Galactic map into a dynamical one",
    ha="center",
    va="center",
    color="#A9C9D8",
    fontsize=11,
    alpha=0,
    zorder=100
)


final_equation = fig.text(
    0.5,
    0.175,
    "X • Y • Z    +    VX • VY • VZ    =    6D PHASE SPACE",
    ha="center",
    va="center",
    color="#C6F4FF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_note = fig.text(
    0.5,
    0.115,
    "SCHEMATIC VISUALISATION — POSITIONS, VELOCITIES AND STREAM GEOMETRY ARE ILLUSTRATIVE",
    ha="center",
    va="center",
    color="#7795A4",
    fontsize=8.5,
    alpha=0,
    zorder=100
)


# ============================================================
# CLOCK
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)
clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_radius = 0.82


dial_outer = Circle(
    (0, 0),
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    (0, 0),
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(dial_outer)
clock_ax.add_patch(dial_inner)


for i in range(12):

    a = 2 * np.pi * i / 12

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = dial_radius * 0.83

    x1 = r1 * np.sin(a)
    y1 = r1 * np.cos(a)

    x2 = r2 * np.sin(a)
    y2 = r2 * np.cos(a)

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = dial_radius * 0.56

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(dial_pin)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS
    t = frame / (NFRAMES - 1)


    # ========================================================
    # POSITIONS
    # ========================================================

    pos_p = smoothstep(
        sec / 1.5
    )

    disk_points.set_alpha(
        0.60 * pos_p
    )

    halo_points.set_alpha(
        0.38 * pos_p
    )


    # ========================================================
    # VELOCITIES APPEAR
    # ========================================================

    vel_p = smoothstep(
        (sec - 4.0) / 1.8
    )

    plus_text.set_alpha(
        vel_p
    )

    velocity_label.set_alpha(
        vel_p
    )

    velocity_values.set_alpha(
        vel_p
    )

    sixd_text.set_alpha(
        vel_p
    )


    for arrow in disk_arrows:

        arrow.arrow_patch.set_alpha(
            0.80 * vel_p
        )


    for arrow in halo_arrows:

        arrow.arrow_patch.set_alpha(
            0.62 * vel_p
        )


    # ========================================================
    # STREAM
    # ========================================================

    stream_p = smoothstep(
        (sec - 7.0) / 1.5
    )

    stream_points.set_alpha(
        0.88 * stream_p
    )


    for arrow in stream_arrows:

        arrow.arrow_patch.set_alpha(
            0.92 * stream_p
        )


    stream_label.set_alpha(
        stream_p
    )

    stream_sub.set_alpha(
        stream_p
    )


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 4.0:

        phase_text.set_text(
            "A STAR MAP SHOWS POSITION"
        )

        phase_sub.set_text(
            "Three coordinates describe where each star is"
        )

    elif sec < 8.0:

        phase_text.set_text(
            "NOW ADD VELOCITY"
        )

        phase_sub.set_text(
            "Three more components describe how each star moves"
        )

    else:

        phase_text.set_text(
            "MOTION REVEALS DYNAMICAL STRUCTURE"
        )

        phase_sub.set_text(
            "Related stars can remain coherent in phase space"
        )


    # ========================================================
    # SUN PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2*np.pi*sec/2.8
        )
    )

    sun_glow.set_alpha(
        0.10 +
        0.08*pulse
    )


    # ========================================================
    # FINAL TRANSITION
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = 1.0 - scene2


    ax.set_alpha(
        scene1_alpha
    )

    info_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_title.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # FINAL SCENE
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(True)
        result_bg.set_alpha(
            0.98 * scene2
        )

        final_left.set_visible(True)
        final_right.set_visible(True)

        final_left.set_alpha(
            scene2
        )

        final_right.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(False)
        final_left.set_visible(False)
        final_right.set_visible(False)


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(scene2)
    final_subheader.set_alpha(scene2)
    final_equation.set_alpha(scene2)
    final_note.set_alpha(scene2)


    # ========================================================
    # CLOCK
    # ========================================================

    dial_angle = 2 * np.pi * t

    hand_x = (
        hand_length *
        np.sin(dial_angle)
    )

    hand_y = (
        hand_length *
        np.cos(dial_angle)
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# От карты звёзд к фазовому пространству

Обычная трёхмерная карта отвечает на вопрос: где находится звезда? Для этого достаточно трёх пространственных координат — X, Y и Z. Но такая карта остаётся статичной: она ничего не говорит о том, как звёзды движутся по Галактике.

Полное шестимерное фазовое пространство добавляет ещё три величины — компоненты скорости VX, VY и VZ. Новый объединённый каталог содержит полную 6D-информацию для 32 552 876 звёзд, объединяя астрометрию Gaia DR3 с расстояниями и лучевыми скоростями крупных спектроскопических обзоров.

Добавление скоростей принципиально меняет картину. Звёзды диска демонстрируют преимущественно согласованное галактическое вращение, тогда как кинематика гало значительно разнообразнее. А звёзды, которые в обычной пространственной карте могут выглядеть лишь частью разреженного фона, способны образовывать связанные структуры в фазовом пространстве.

Именно поэтому 6D-данные особенно полезны для поиска и исследования звёздных потоков, скоплений, спутниковых галактик и других динамически связанных популяций. В статье возможности объединённого каталога проверяются, в частности, на скоплениях, карликовых галактиках и потоке Стрельца.

3D показывает, где находится звезда.
6D показывает, куда она движется.

Положения звёзд, направления скоростей и изображённый поток в анимации схематические и предназначены для объяснения принципа фазового пространства.

# From a Star Map to Phase Space

A conventional three-dimensional map answers one question: where is a star? Three spatial coordinates — X, Y, and Z — are sufficient to describe its position. But such a map remains static: it does not tell us how stars move through the Galaxy.

Full six-dimensional phase space adds three more quantities — the velocity components VX, VY, and VZ. The new homogenised catalogue provides complete 6D information for 32,552,876 stars, combining Gaia DR3 astrometry with distances and line-of-sight velocities from major spectroscopic surveys.

Adding velocity fundamentally changes the picture. Disk stars predominantly participate in coherent Galactic rotation, while halo kinematics are much more diverse. Stars that may appear to be part of a diffuse background in an ordinary spatial map can form related structures once their motions are included.

This is why 6D information is particularly valuable for identifying and studying stellar streams, clusters, satellite galaxies, and other dynamically related populations. The study validates the combined catalogue using structures including stellar clusters, dwarf galaxies, and the Sagittarius stream.

3D tells us where a star is.
6D tells us where it is going.

The stellar positions, velocity directions, and stream shown in the animation are schematic and are intended to illustrate the concept of phase space.